In [1]:
import torch

print("PyTorch is available!")
print("PyTorch version:", torch.__version__)

PyTorch is available!
PyTorch version: 2.11.0+cpu


In [2]:
import os

print(os.path.exists("guardian_weights.json"))

True


In [3]:
import json

with open("guardian_weights.json", "r") as f:
    weights = json.load(f)

print("Saved components:")
print(weights.keys())

Saved components:
dict_keys(['W1', 'b1', 'W2', 'b2'])


In [4]:
print("W1 first row:")
print(weights["W1"][0])

print("\nb1:")
print(weights["b1"])

print("\nW2 first row:")
print(weights["W2"][0])

print("\nb2:")
print(weights["b2"])

W1 first row:
[2.8040857538180726, -0.22352442523064303, -0.3967061801479005, -1.9007098882790414, -0.4158907982230808, 1.9945224646733244, -0.2193868076646294, -0.6496214967116979, 0.8939155492036421, -1.9437871450452335, -0.586666953951453, 1.8231782653537756, 2.191194925044045, -1.733951076779032, -0.50247757460757, -1.5789791458246463, -1.6292767176348306, 1.907088661126322]

b1:
[0.4588503788407335, -0.2617426950895906, 1.2187554482975864, -0.07272003131993579, 0.6872728088271511, 0.037310325016387515]

W2 first row:
[3.428280552070877, 2.0135783383435215, -4.643285556829542, -5.523961652952157, 0.7475466091115087, 0.39661969618964443]

b2:
[-1.800505252681807, -0.948592729112229, -3.3364136433816913, -1.60610720616707, -1.7292060478913067, -2.3870387055193003, -3.5000681565260856, -0.5086001910157161, -2.039839931931292, -0.18703746683559247, -0.28047981877494477, -2.5526187676505887, -1.6494113596619362, -0.9250304695424421, -1.9183690696176037, -1.5798648246594607, 2.3112477561

In [5]:
import torch

W1 = torch.tensor(weights["W1"], dtype=torch.float32)
b1 = torch.tensor(weights["b1"], dtype=torch.float32)
W2 = torch.tensor(weights["W2"], dtype=torch.float32)
b2 = torch.tensor(weights["b2"], dtype=torch.float32)

print("W1 shape:", W1.shape)
print("b1 shape:", b1.shape)
print("W2 shape:", W2.shape)
print("b2 shape:", b2.shape)

W1 shape: torch.Size([6, 18])
b1 shape: torch.Size([6])
W2 shape: torch.Size([18, 6])
b2 shape: torch.Size([18])


In [6]:
import torch

class GuardianAutoencoder(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = torch.nn.Linear(18, 6)
        self.decoder = torch.nn.Linear(6, 18)

    def forward(self, x):
        x = torch.sigmoid(self.encoder(x))
        x = torch.sigmoid(self.decoder(x))
        return x

In [7]:
model = GuardianAutoencoder()

print(model)

GuardianAutoencoder(
  (encoder): Linear(in_features=18, out_features=6, bias=True)
  (decoder): Linear(in_features=6, out_features=18, bias=True)
)


In [8]:
with torch.no_grad():
    model.encoder.weight.copy_(W1)
    model.encoder.bias.copy_(b1)

    model.decoder.weight.copy_(W2)
    model.decoder.bias.copy_(b2)

print("Trained Guardian weights loaded into PyTorch model.")

Trained Guardian weights loaded into PyTorch model.


In [9]:
print("PyTorch weight shapes:")
print("Encoder weight:", model.encoder.weight.shape)
print("Encoder bias:", model.encoder.bias.shape)
print("Decoder weight:", model.decoder.weight.shape)
print("Decoder bias:", model.decoder.bias.shape)

PyTorch weight shapes:
Encoder weight: torch.Size([6, 18])
Encoder bias: torch.Size([6])
Decoder weight: torch.Size([18, 6])
Decoder bias: torch.Size([18])


In [10]:
test_input = torch.tensor(
    [[
        1.0, 0.0, 0.0, 1.0, 0.0, 0.0,
        1.0, 0.0, 0.0, 1.0, 0.0, 0.0,
        1.0, 0.0, 0.0, 1.0, 0.0, 0.0
    ]],
    dtype=torch.float32
)

print("Test input shape:", test_input.shape)

Test input shape: torch.Size([1, 18])


In [11]:
with torch.no_grad():
    pytorch_output = model(test_input)

print("PyTorch output shape:", pytorch_output.shape)
print("PyTorch reconstruction:")
print(pytorch_output)

PyTorch output shape: torch.Size([1, 18])
PyTorch reconstruction:
tensor([[1.1571e-01, 9.6991e-03, 8.5571e-06, 5.5276e-02, 6.3405e-03, 1.9832e-03,
         3.9347e-02, 4.3949e-03, 2.4699e-03, 7.0631e-02, 4.7597e-05, 2.5584e-03,
         1.1390e-01, 1.6360e-01, 6.1755e-03, 5.3717e-02, 9.9795e-01, 2.2340e-03]])


In [12]:
import math

def original_sigmoid(x):
    x = max(-50.0, min(50.0, x))
    return 1.0 / (1.0 + math.exp(-x))


def original_forward(x):
    # Encoder
    hidden = []

    for i in range(6):
        total = b1[i].item()

        for j in range(18):
            total += W1[i][j].item() * x[j]

        hidden.append(original_sigmoid(total))

    # Decoder
    output = []

    for i in range(18):
        total = b2[i].item()

        for j in range(6):
            total += W2[i][j].item() * hidden[j]

        output.append(original_sigmoid(total))

    return output

In [13]:
original_output = original_forward(test_input[0].tolist())

print("Original Python reconstruction:")
print(original_output)

Original Python reconstruction:
[0.11571157022754652, 0.009699153440280954, 8.557092149225296e-06, 0.05527568034829596, 0.006340482970466663, 0.0019832192336779126, 0.03934671203971535, 0.004394888888816526, 0.0024699296946331466, 0.0706305012007658, 4.7596730501029455e-05, 0.002558400100196247, 0.1138979583025531, 0.16360401076835468, 0.006175489617556733, 0.0537170327653755, 0.9979543050148257, 0.002234037637860594]


In [14]:
original_tensor = torch.tensor(
    [original_output],
    dtype=torch.float32
)

difference = torch.abs(original_tensor - pytorch_output)

print("Maximum absolute difference:", difference.max().item())
print("Average absolute difference:", difference.mean().item())

print("\nOutputs match within tolerance:",
      torch.allclose(
          original_tensor,
          pytorch_output,
          rtol=1e-5,
          atol=1e-6
      ))

Maximum absolute difference: 7.450580596923828e-08
Average absolute difference: 1.7077480762850428e-08

Outputs match within tolerance: True


In [15]:
def pytorch_reconstruction_error(event_features):
    x = torch.tensor(
        [event_features],
        dtype=torch.float32
    )

    with torch.no_grad():
        output = model(x)

    error = torch.mean((output - x) ** 2)

    return error.item()

In [16]:
# Recreate the original v0.9 feature encoding
# for behavioral testing

normal_events = [

    {
        "source": "HMI-01",
        "destination": "PLC-01",
        "information": "process_read",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "HMI-01",
        "destination": "PLC-01",
        "information": "process_read",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "HMI-01",
        "destination": "PLC-02",
        "information": "process_read",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "PLC-01",
        "destination": "HMI-01",
        "information": "process_status",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "PLC-01",
        "destination": "HMI-01",
        "information": "process_status",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "PLC-02",
        "destination": "HMI-01",
        "information": "process_status",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "PLC-01",
        "destination": "PLC-02",
        "information": "controller_state",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "PLC-01",
        "destination": "PLC-02",
        "information": "controller_state",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "PLC-02",
        "destination": "PLC-01",
        "information": "controller_state",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "PLC-02",
        "destination": "PLC-01",
        "information": "controller_state",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "SCADA-01",
        "destination": "PLC-01",
        "information": "supervisory_command",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "PLC-01",
        "destination": "SCADA-01",
        "information": "process_status",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "PLC-01",
        "destination": "HIST-01",
        "information": "process_data",
        "protocol": "OPC-UA",
        "port": 4840
    },

    {
        "source": "PLC-01",
        "destination": "HIST-01",
        "information": "process_data",
        "protocol": "OPC-UA",
        "port": 4840
    },

    {
        "source": "PLC-01",
        "destination": "HIST-01",
        "information": "process_data",
        "protocol": "OPC-UA",
        "port": 4840
    },

    {
        "source": "PLC-01",
        "destination": "HIST-01",
        "information": "process_data",
        "protocol": "OPC-UA",
        "port": 4840
    },

    {
        "source": "PLC-02",
        "destination": "HIST-01",
        "information": "process_data",
        "protocol": "OPC-UA",
        "port": 4840
    },

    {
        "source": "PLC-02",
        "destination": "HIST-01",
        "information": "process_data",
        "protocol": "OPC-UA",
        "port": 4840
    },

    {
        "source": "PLC-02",
        "destination": "HIST-01",
        "information": "process_data",
        "protocol": "OPC-UA",
        "port": 4840
    },

    {
        "source": "SIS-01",
        "destination": "SCADA-01",
        "information": "safety_status",
        "protocol": "Modbus/TCP",
        "port": 502
    },

    {
        "source": "SIS-01",
        "destination": "SCADA-01",
        "information": "safety_status",
        "protocol": "Modbus/TCP",
        "port": 502
    }
]


# Original v0.9 timestamps

normal_timestamps = [
    0,5,10,15,20,25,30,35,40,45,50,55,
    65,75,85,95,105,115,125,135,140
]


# Original v0.9 traffic volumes

normal_volumes = [
    180,180,180,160,160,160,140,140,140,140,
    210,190,820,820,820,820,820,820,820,120,120
]


# Attach timestamps and volumes to the events

for index, event in enumerate(normal_events):
    event["timestamp"] = normal_timestamps[index]
    event["bytes"] = normal_volumes[index]


# Build the exact feature vocabularies used by v0.9

sources = sorted(
    set(event["source"] for event in normal_events)
)

destinations = sorted(
    set(event["destination"] for event in normal_events)
)

information_types = sorted(
    set(event["information"] for event in normal_events)
)

protocols = sorted(
    set(
        (event["protocol"], event["port"])
        for event in normal_events
    )
)


# Original v0.9 one-hot encoder

def one_hot(value, known_values):

    vector = []

    for item in known_values:

        if value == item:
            vector.append(1.0)

        else:
            vector.append(0.0)

    return vector


# Original v0.9 event-to-feature conversion

def event_to_features(event):

    features = []

    features += one_hot(
        event["source"],
        sources
    )

    features += one_hot(
        event["destination"],
        destinations
    )

    features += one_hot(
        event["information"],
        information_types
    )

    protocol_key = (
        event["protocol"],
        event["port"]
    )

    features += one_hot(
        protocol_key,
        protocols
    )

    return features


# Create the original v0.9 training data

training_data = [
    event_to_features(event)
    for event in normal_events
]

input_size = len(training_data[0])


print("Feature encoder recreated.")
print("Sources:", sources)
print("Destinations:", destinations)
print("Information types:", information_types)
print("Protocols:", protocols)
print("Input size:", input_size)

Feature encoder recreated.
Sources: ['HMI-01', 'PLC-01', 'PLC-02', 'SCADA-01', 'SIS-01']
Destinations: ['HIST-01', 'HMI-01', 'PLC-01', 'PLC-02', 'SCADA-01']
Information types: ['controller_state', 'process_data', 'process_read', 'process_status', 'safety_status', 'supervisory_command']
Protocols: [('Modbus/TCP', 502), ('OPC-UA', 4840)]
Input size: 18


In [17]:
# Test a known-normal OT event

normal_test_event = {
    "source": "PLC-01",
    "destination": "HIST-01",
    "information": "process_data",
    "protocol": "OPC-UA",
    "port": 4840
}

test_features = event_to_features(normal_test_event)

pytorch_score = pytorch_reconstruction_error(test_features)

print("Test event:")
print(normal_test_event)

print("\nFeature vector:")
print(test_features)

print("\nPyTorch reconstruction error:")
print(pytorch_score)

Test event:
{'source': 'PLC-01', 'destination': 'HIST-01', 'information': 'process_data', 'protocol': 'OPC-UA', 'port': 4840}

Feature vector:
[0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]

PyTorch reconstruction error:
0.0001249280321644619


In [18]:
# Calculate reconstruction errors for all original normal events
# using the original v0.9 weights and feature encoding

training_errors = []

for event in normal_events:

    features = event_to_features(event)

    original_output = original_forward(features)

    error = sum(
        (original_output[i] - features[i]) ** 2
        for i in range(len(features))
    ) / len(features)

    training_errors.append(error)


maximum_training_error = max(training_errors)

threshold = maximum_training_error * 1.5

print("Maximum training reconstruction error:")
print(maximum_training_error)

print("\nOriginal v0.9 anomaly threshold:")
print(threshold)

print("\nNormal test event reconstruction error:")
print(pytorch_score)

print("\nClassified as anomaly:",
      pytorch_score > threshold)

Maximum training reconstruction error:
0.0006978409170010442

Original v0.9 anomaly threshold:
0.0010467613755015663

Normal test event reconstruction error:
0.0001249280321644619

Classified as anomaly: False


In [19]:
# Test suspicious reverse-direction behavior

suspicious_event = {
    "source": "HIST-01",
    "destination": "PLC-01",
    "information": "process_data",
    "protocol": "OPC-UA",
    "port": 4840
}

suspicious_features = event_to_features(suspicious_event)

suspicious_score = pytorch_reconstruction_error(
    suspicious_features
)

print("Suspicious event:")
print(suspicious_event)

print("\nFeature vector:")
print(suspicious_features)

print("\nPyTorch reconstruction error:")
print(suspicious_score)

print("\nOriginal v0.9 threshold:")
print(threshold)

print("\nClassified as anomaly:",
      suspicious_score > threshold)

Suspicious event:
{'source': 'HIST-01', 'destination': 'PLC-01', 'information': 'process_data', 'protocol': 'OPC-UA', 'port': 4840}

Feature vector:
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]

PyTorch reconstruction error:
0.1426754593849182

Original v0.9 threshold:
0.0010467613755015663

Classified as anomaly: True


In [20]:
# Test known devices with a different information type

new_information_event = {
    "source": "PLC-01",
    "destination": "HIST-01",
    "information": "process_status",
    "protocol": "OPC-UA",
    "port": 4840
}

new_information_features = event_to_features(
    new_information_event
)

new_information_score = pytorch_reconstruction_error(
    new_information_features
)

print("Test event:")
print(new_information_event)

print("\nFeature vector:")
print(new_information_features)

print("\nPyTorch reconstruction error:")
print(new_information_score)

print("\nOriginal v0.9 threshold:")
print(threshold)

print("\nClassified as anomaly:",
      new_information_score > threshold)

Test event:
{'source': 'PLC-01', 'destination': 'HIST-01', 'information': 'process_status', 'protocol': 'OPC-UA', 'port': 4840}

Feature vector:
[0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0]

PyTorch reconstruction error:
0.09642583131790161

Original v0.9 threshold:
0.0010467613755015663

Classified as anomaly: True


In [21]:
# Test a new protocol on a known device relationship

new_protocol_event = {
    "source": "PLC-01",
    "destination": "HIST-01",
    "information": "process_data",
    "protocol": "Modbus/TCP",
    "port": 502
}

new_protocol_features = event_to_features(
    new_protocol_event
)

new_protocol_score = pytorch_reconstruction_error(
    new_protocol_features
)

print("Test event:")
print(new_protocol_event)

print("\nFeature vector:")
print(new_protocol_features)

print("\nPyTorch reconstruction error:")
print(new_protocol_score)

print("\nOriginal v0.9 threshold:")
print(threshold)

print("\nClassified as anomaly:",
      new_protocol_score > threshold)

Test event:
{'source': 'PLC-01', 'destination': 'HIST-01', 'information': 'process_data', 'protocol': 'Modbus/TCP', 'port': 502}

Feature vector:
[0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]

PyTorch reconstruction error:
0.09560397267341614

Original v0.9 threshold:
0.0010467613755015663

Classified as anomaly: True


In [22]:
# Test a completely new device relationship

new_relationship_event = {
    "source": "HMI-01",
    "destination": "HIST-01",
    "information": "process_data",
    "protocol": "OPC-UA",
    "port": 4840
}

new_relationship_features = event_to_features(
    new_relationship_event
)

new_relationship_score = pytorch_reconstruction_error(
    new_relationship_features
)

print("Test event:")
print(new_relationship_event)

print("\nFeature vector:")
print(new_relationship_features)

print("\nPyTorch reconstruction error:")
print(new_relationship_score)

print("\nOriginal v0.9 threshold:")
print(threshold)

print("\nClassified as anomaly:",
      new_relationship_score > threshold)

Test event:
{'source': 'HMI-01', 'destination': 'HIST-01', 'information': 'process_data', 'protocol': 'OPC-UA', 'port': 4840}

Feature vector:
[1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]

PyTorch reconstruction error:
0.06498560309410095

Original v0.9 threshold:
0.0010467613755015663

Classified as anomaly: True


In [23]:
# Test a known information type in a new context

new_context_event = {
    "source": "PLC-02",
    "destination": "HIST-01",
    "information": "process_read",
    "protocol": "OPC-UA",
    "port": 4840
}

new_context_features = event_to_features(
    new_context_event
)

new_context_score = pytorch_reconstruction_error(
    new_context_features
)

print("Test event:")
print(new_context_event)

print("\nFeature vector:")
print(new_context_features)

print("\nPyTorch reconstruction error:")
print(new_context_score)

print("\nOriginal v0.9 threshold:")
print(threshold)

print("\nClassified as anomaly:",
      new_context_score > threshold)

Test event:
{'source': 'PLC-02', 'destination': 'HIST-01', 'information': 'process_read', 'protocol': 'OPC-UA', 'port': 4840}

Feature vector:
[0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0]

PyTorch reconstruction error:
0.10941316187381744

Original v0.9 threshold:
0.0010467613755015663

Classified as anomaly: True


In [24]:
# Test an unknown source device

unknown_device_event = {
    "source": "NEW-DEVICE",
    "destination": "PLC-01",
    "information": "process_status",
    "protocol": "Modbus/TCP",
    "port": 502
}

unknown_device_features = event_to_features(
    unknown_device_event
)

unknown_device_score = pytorch_reconstruction_error(
    unknown_device_features
)

print("Test event:")
print(unknown_device_event)

print("\nFeature vector:")
print(unknown_device_features)

print("\nPyTorch reconstruction error:")
print(unknown_device_score)

print("\nOriginal v0.9 threshold:")
print(threshold)

print("\nClassified as anomaly:",
      unknown_device_score > threshold)

Test event:
{'source': 'NEW-DEVICE', 'destination': 'PLC-01', 'information': 'process_status', 'protocol': 'Modbus/TCP', 'port': 502}

Feature vector:
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]

PyTorch reconstruction error:
0.09530343115329742

Original v0.9 threshold:
0.0010467613755015663

Classified as anomaly: True


In [25]:
# ======================================================================
# OT GUARDIAN v0.9
# COMPLETE PYTORCH BEHAVIORAL VALIDATION
# ======================================================================
#
# Purpose:
# Compare the deployed PyTorch autoencoder behavior against
# the original v0.9 anomaly threshold across multiple OT scenarios.
#
# IMPORTANT:
# - No training is performed here.
# - No weights are changed.
# - The existing PyTorch model "model" is used.
# - The existing v0.9 feature encoder is used.
# ======================================================================


# ======================================================================
# TEST 0 — VERIFY ENVIRONMENT
# ======================================================================

print("=" * 70)
print("TEST 0 — ENVIRONMENT / MODEL CHECK")
print("=" * 70)

print("Expected input size: 18")
print("Actual input size:  ", input_size)

print("\nPyTorch model:")
print(model)

print("\nModel verification:")
if input_size == 18:
    print("PASS: Feature input size is 18.")
else:
    print("FAIL: Feature input size is NOT 18.")

print("\n" + "=" * 70)


# ======================================================================
# TEST 1 — KNOWN NORMAL EVENT
# ======================================================================

print("TEST 1 — KNOWN NORMAL EVENT")
print("=" * 70)

normal_event = {
    "source": "PLC-01",
    "destination": "HIST-01",
    "information": "process_data",
    "protocol": "OPC-UA",
    "port": 4840
}

normal_features = event_to_features(normal_event)
normal_score = pytorch_reconstruction_error(normal_features)

print("Event:")
print(normal_event)

print("\nFeature vector:")
print(normal_features)

print("\nReconstruction error:")
print(normal_score)

print("\nThreshold:")
print(threshold)

normal_result = normal_score > threshold

print("\nClassification:")
print("ANOMALY" if normal_result else "NORMAL")

if not normal_result:
    print("PASS: Known-normal event remains normal.")
else:
    print("FAIL: Known-normal event was incorrectly flagged.")


# ======================================================================
# TEST 2 — REVERSE DIRECTION
# ======================================================================

print("\n" + "=" * 70)
print("TEST 2 — REVERSE DIRECTION")
print("=" * 70)

reverse_event = {
    "source": "HIST-01",
    "destination": "PLC-01",
    "information": "process_data",
    "protocol": "OPC-UA",
    "port": 4840
}

reverse_features = event_to_features(reverse_event)
reverse_score = pytorch_reconstruction_error(reverse_features)

print("Event:")
print(reverse_event)

print("\nReconstruction error:")
print(reverse_score)

print("\nThreshold:")
print(threshold)

reverse_result = reverse_score > threshold

print("\nClassification:")
print("ANOMALY" if reverse_result else "NORMAL")

if reverse_result:
    print("PASS: Reverse-direction behavior detected.")
else:
    print("FAIL: Reverse-direction behavior was not detected.")


# ======================================================================
# TEST 3 — NEW INFORMATION TYPE
# ======================================================================

print("\n" + "=" * 70)
print("TEST 3 — NEW INFORMATION TYPE")
print("=" * 70)

new_information_event = {
    "source": "PLC-01",
    "destination": "HIST-01",
    "information": "process_status",
    "protocol": "OPC-UA",
    "port": 4840
}

new_information_features = event_to_features(
    new_information_event
)

new_information_score = pytorch_reconstruction_error(
    new_information_features
)

print("Event:")
print(new_information_event)

print("\nReconstruction error:")
print(new_information_score)

print("\nThreshold:")
print(threshold)

new_information_result = new_information_score > threshold

print("\nClassification:")
print(
    "ANOMALY"
    if new_information_result
    else "NORMAL"
)

if new_information_result:
    print("PASS: New information type detected.")
else:
    print("FAIL: New information type was not detected.")


# ======================================================================
# TEST 4 — NEW PROTOCOL
# ======================================================================

print("\n" + "=" * 70)
print("TEST 4 — NEW PROTOCOL")
print("=" * 70)

new_protocol_event = {
    "source": "PLC-01",
    "destination": "HIST-01",
    "information": "process_data",
    "protocol": "Modbus/TCP",
    "port": 502
}

new_protocol_features = event_to_features(
    new_protocol_event
)

new_protocol_score = pytorch_reconstruction_error(
    new_protocol_features
)

print("Event:")
print(new_protocol_event)

print("\nReconstruction error:")
print(new_protocol_score)

print("\nThreshold:")
print(threshold)

new_protocol_result = new_protocol_score > threshold

print("\nClassification:")
print(
    "ANOMALY"
    if new_protocol_result
    else "NORMAL"
)

if new_protocol_result:
    print("PASS: New protocol detected.")
else:
    print("FAIL: New protocol was not detected.")


# ======================================================================
# TEST 5 — NEW DEVICE RELATIONSHIP
# ======================================================================

print("\n" + "=" * 70)
print("TEST 5 — NEW DEVICE RELATIONSHIP")
print("=" * 70)

new_relationship_event = {
    "source": "HMI-01",
    "destination": "HIST-01",
    "information": "process_data",
    "protocol": "OPC-UA",
    "port": 4840
}

new_relationship_features = event_to_features(
    new_relationship_event
)

new_relationship_score = pytorch_reconstruction_error(
    new_relationship_features
)

print("Event:")
print(new_relationship_event)

print("\nReconstruction error:")
print(new_relationship_score)

print("\nThreshold:")
print(threshold)

new_relationship_result = (
    new_relationship_score > threshold
)

print("\nClassification:")
print(
    "ANOMALY"
    if new_relationship_result
    else "NORMAL"
)

if new_relationship_result:
    print("PASS: New device relationship detected.")
else:
    print("FAIL: New device relationship was not detected.")


# ======================================================================
# TEST 6 — KNOWN INFORMATION IN NEW CONTEXT
# ======================================================================

print("\n" + "=" * 70)
print("TEST 6 — KNOWN INFORMATION IN NEW CONTEXT")
print("=" * 70)

new_context_event = {
    "source": "PLC-02",
    "destination": "HIST-01",
    "information": "process_read",
    "protocol": "OPC-UA",
    "port": 4840
}

new_context_features = event_to_features(
    new_context_event
)

new_context_score = pytorch_reconstruction_error(
    new_context_features
)

print("Event:")
print(new_context_event)

print("\nReconstruction error:")
print(new_context_score)

print("\nThreshold:")
print(threshold)

new_context_result = new_context_score > threshold

print("\nClassification:")
print(
    "ANOMALY"
    if new_context_result
    else "NORMAL"
)

if new_context_result:
    print("PASS: Abnormal feature combination detected.")
else:
    print("FAIL: Abnormal feature combination was not detected.")


# ======================================================================
# TEST 7 — UNKNOWN SOURCE DEVICE
# ======================================================================

print("\n" + "=" * 70)
print("TEST 7 — UNKNOWN SOURCE DEVICE")
print("=" * 70)

unknown_source_event = {
    "source": "NEW-DEVICE",
    "destination": "PLC-01",
    "information": "process_status",
    "protocol": "Modbus/TCP",
    "port": 502
}

unknown_source_features = event_to_features(
    unknown_source_event
)

unknown_source_score = pytorch_reconstruction_error(
    unknown_source_features
)

print("Event:")
print(unknown_source_event)

print("\nFeature vector:")
print(unknown_source_features)

print("\nReconstruction error:")
print(unknown_source_score)

print("\nThreshold:")
print(threshold)

unknown_source_result = unknown_source_score > threshold

print("\nClassification:")
print(
    "ANOMALY"
    if unknown_source_result
    else "NORMAL"
)

if unknown_source_result:
    print("PASS: Unknown source produced anomalous behavior.")
else:
    print("FAIL: Unknown source was not flagged by AI.")


# ======================================================================
# TEST 8 — UNKNOWN DESTINATION DEVICE
# ======================================================================

print("\n" + "=" * 70)
print("TEST 8 — UNKNOWN DESTINATION DEVICE")
print("=" * 70)

unknown_destination_event = {
    "source": "PLC-01",
    "destination": "NEW-DEVICE",
    "information": "process_data",
    "protocol": "OPC-UA",
    "port": 4840
}

unknown_destination_features = event_to_features(
    unknown_destination_event
)

unknown_destination_score = pytorch_reconstruction_error(
    unknown_destination_features
)

print("Event:")
print(unknown_destination_event)

print("\nFeature vector:")
print(unknown_destination_features)

print("\nReconstruction error:")
print(unknown_destination_score)

print("\nThreshold:")
print(threshold)

unknown_destination_result = (
    unknown_destination_score > threshold
)

print("\nClassification:")
print(
    "ANOMALY"
    if unknown_destination_result
    else "NORMAL"
)

if unknown_destination_result:
    print("PASS: Unknown destination produced anomalous behavior.")
else:
    print("FAIL: Unknown destination was not flagged by AI.")


# ======================================================================
# TEST 9 — MULTIPLE ANOMALOUS FEATURES
# ======================================================================

print("\n" + "=" * 70)
print("TEST 9 — MULTIPLE SIMULTANEOUS ANOMALIES")
print("=" * 70)

multiple_anomalies_event = {
    "source": "NEW-DEVICE",
    "destination": "HIST-01",
    "information": "process_status",
    "protocol": "Modbus/TCP",
    "port": 502
}

multiple_anomalies_features = event_to_features(
    multiple_anomalies_event
)

multiple_anomalies_score = pytorch_reconstruction_error(
    multiple_anomalies_features
)

print("Event:")
print(multiple_anomalies_event)

print("\nFeature vector:")
print(multiple_anomalies_features)

print("\nReconstruction error:")
print(multiple_anomalies_score)

print("\nThreshold:")
print(threshold)

multiple_anomalies_result = (
    multiple_anomalies_score > threshold
)

print("\nClassification:")
print(
    "ANOMALY"
    if multiple_anomalies_result
    else "NORMAL"
)

if multiple_anomalies_result:
    print("PASS: Multiple anomalous features detected.")
else:
    print("FAIL: Multiple anomalous features were not detected.")


# ======================================================================
# FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("FINAL BEHAVIORAL VALIDATION SUMMARY")
print("=" * 70)

results = {
    "Test 1 - Known normal": not normal_result,
    "Test 2 - Reverse direction": reverse_result,
    "Test 3 - New information": new_information_result,
    "Test 4 - New protocol": new_protocol_result,
    "Test 5 - New relationship": new_relationship_result,
    "Test 6 - New context": new_context_result,
    "Test 7 - Unknown source": unknown_source_result,
    "Test 8 - Unknown destination": unknown_destination_result,
    "Test 9 - Multiple anomalies": multiple_anomalies_result
}

passed = 0
failed = 0

for test_name, result in results.items():

    if result:
        print("PASS:", test_name)
        passed += 1

    else:
        print("FAIL:", test_name)
        failed += 1


print("\n" + "-" * 70)

print("Tests passed:", passed)
print("Tests failed:", failed)

if failed == 0:
    print("\nOVERALL RESULT: ALL BEHAVIORAL TESTS PASSED")
else:
    print("\nOVERALL RESULT: SOME TESTS FAILED")
    print("Check the individual test section above.")

print("=" * 70)


TEST 0 — ENVIRONMENT / MODEL CHECK
Expected input size: 18
Actual input size:   18

PyTorch model:
GuardianAutoencoder(
  (encoder): Linear(in_features=18, out_features=6, bias=True)
  (decoder): Linear(in_features=6, out_features=18, bias=True)
)

Model verification:
PASS: Feature input size is 18.

TEST 1 — KNOWN NORMAL EVENT
Event:
{'source': 'PLC-01', 'destination': 'HIST-01', 'information': 'process_data', 'protocol': 'OPC-UA', 'port': 4840}

Feature vector:
[0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]

Reconstruction error:
0.0001249280321644619

Threshold:
0.0010467613755015663

Classification:
NORMAL
PASS: Known-normal event remains normal.

TEST 2 — REVERSE DIRECTION
Event:
{'source': 'HIST-01', 'destination': 'PLC-01', 'information': 'process_data', 'protocol': 'OPC-UA', 'port': 4840}

Reconstruction error:
0.1426754593849182

Threshold:
0.0010467613755015663

Classification:
ANOMALY
PASS: Reverse-direction behavior detected.

TES

In [26]:
# ======================================================================
# RESTORE ORIGINAL v0.9 GUARDIAN INVESTIGATION STATE
# ======================================================================

print("=" * 70)
print("RESTORING ORIGINAL v0.9 INVESTIGATION STATE")
print("=" * 70)

# ----------------------------------------------------------------------
# CONTEXT KNOWLEDGE
# ----------------------------------------------------------------------

known_relationships = set()
known_directions = set()

known_information_by_direction = set()
known_protocols_by_direction = set()

known_information_types = set()
known_protocol_types = set()

for event in normal_events:

    source = event["source"]
    destination = event["destination"]

    relationship = frozenset(
        (source, destination)
    )

    direction = (
        source,
        destination
    )

    information_key = (
        source,
        destination,
        event["information"]
    )

    protocol_key = (
        source,
        destination,
        event["protocol"],
        event["port"]
    )

    known_relationships.add(
        relationship
    )

    known_directions.add(
        direction
    )

    known_information_by_direction.add(
        information_key
    )

    known_protocols_by_direction.add(
        protocol_key
    )

    known_information_types.add(
        event["information"]
    )

    known_protocol_types.add(
        (
            event["protocol"],
            event["port"]
        )
    )


# ----------------------------------------------------------------------
# TEMPORAL KNOWLEDGE
# ----------------------------------------------------------------------

normal_time_gaps = []

for i in range(1, len(normal_timestamps)):

    gap = (
        normal_timestamps[i]
        - normal_timestamps[i - 1]
    )

    normal_time_gaps.append(
        gap
    )

average_time_gap = (
    sum(normal_time_gaps)
    / len(normal_time_gaps)
)

maximum_time_gap = max(
    normal_time_gaps
)

time_gap_threshold = (
    maximum_time_gap * 2
)


# ----------------------------------------------------------------------
# VOLUME KNOWLEDGE
# ----------------------------------------------------------------------

average_volume = (
    sum(normal_volumes)
    / len(normal_volumes)
)

minimum_volume = min(
    normal_volumes
)

maximum_volume = max(
    normal_volumes
)

volume_upper_threshold = (
    maximum_volume * 1.5
)

volume_lower_threshold = (
    minimum_volume * 0.5
)


# ----------------------------------------------------------------------
# VERIFY RESTORED STATE
# ----------------------------------------------------------------------

print("\nContext knowledge:")
print(
    "Known relationships:",
    len(known_relationships)
)

print(
    "Known directions:",
    len(known_directions)
)

print(
    "Known information types:",
    len(known_information_types)
)

print(
    "Known protocol types:",
    len(known_protocol_types)
)

print("\nTemporal knowledge:")
print(
    "Average time gap:",
    average_time_gap
)

print(
    "Maximum time gap:",
    maximum_time_gap
)

print(
    "Time gap threshold:",
    time_gap_threshold
)

print("\nVolume knowledge:")
print(
    "Average volume:",
    average_volume
)

print(
    "Minimum volume:",
    minimum_volume
)

print(
    "Maximum volume:",
    maximum_volume
)

print(
    "Upper volume threshold:",
    volume_upper_threshold
)

print(
    "Lower volume threshold:",
    volume_lower_threshold
)

print("\nPASS: Original v0.9 investigation state restored.")

RESTORING ORIGINAL v0.9 INVESTIGATION STATE

Context knowledge:
Known relationships: 7
Known directions: 11
Known information types: 6
Known protocol types: 2

Temporal knowledge:
Average time gap: 7.0
Maximum time gap: 10
Time gap threshold: 20

Volume knowledge:
Average volume: 379.04761904761904
Minimum volume: 120
Maximum volume: 820
Upper volume threshold: 1230.0
Lower volume threshold: 60.0

PASS: Original v0.9 investigation state restored.


In [27]:
# ======================================================================
# RESTORE ORIGINAL v0.9 DEVICE MAP
# ======================================================================

devices = {
    "PLC-01": {
        "level": 1,
        "ip": "10.0.1.10"
    },

    "PLC-02": {
        "level": 1,
        "ip": "10.0.1.11"
    },

    "SIS-01": {
        "level": 1,
        "ip": "10.0.1.12"
    },

    "HMI-01": {
        "level": 2,
        "ip": "10.0.2.10"
    },

    "SCADA-01": {
        "level": 3,
        "ip": "10.0.3.10"
    },

    "HIST-01": {
        "level": 3.5,
        "ip": "10.0.35.10"
    }
}

print("=" * 70)
print("DEVICE MAP RESTORED")
print("=" * 70)

print("Known devices:", len(devices))

for name, info in devices.items():
    print(
        name,
        "-> Level", info["level"],
        "| IP", info["ip"]
    )

print("\nPASS: Original v0.9 device map restored.")

DEVICE MAP RESTORED
Known devices: 6
PLC-01 -> Level 1 | IP 10.0.1.10
PLC-02 -> Level 1 | IP 10.0.1.11
SIS-01 -> Level 1 | IP 10.0.1.12
HMI-01 -> Level 2 | IP 10.0.2.10
SCADA-01 -> Level 3 | IP 10.0.3.10
HIST-01 -> Level 3.5 | IP 10.0.35.10

PASS: Original v0.9 device map restored.


In [28]:
# ======================================================================
# OT GUARDIAN v0.9
# FULL GUARDIAN VERIFICATION
# PyTorch AI + Context + Temporal + Volume + Decision Fusion
# ======================================================================


print("=" * 70)
print("FULL GUARDIAN VERIFICATION")
print("PyTorch AI + Context + Temporal + Volume + Decision Fusion")
print("=" * 70)


# ======================================================================
# 1. VERIFY THAT THE PYTORCH MODEL IS READY
# ======================================================================

print("\n" + "=" * 70)
print("STEP 1 — PYTORCH MODEL CHECK")
print("=" * 70)

print("Model:")
print(model)

print("\nInput size:", input_size)
print("Expected input size: 18")

if input_size == 18:
    print("PASS: Correct 18-feature input.")
else:
    print("FAIL: Input size is incorrect.")


# ======================================================================
# 2. PYTORCH RECONSTRUCTION ERROR
# ======================================================================
#
# This replaces the original v0.9 "forward()" based AI scoring.
#
# Everything AFTER this function remains the original Guardian logic.
# ======================================================================

def pytorch_guardian_reconstruction_error(event):

    features = event_to_features(event)

    x = torch.tensor(
        [features],
        dtype=torch.float32
    )

    with torch.no_grad():

        output = model(x)

    error = torch.mean(
        (output - x) ** 2
    )

    return error.item()


# ======================================================================
# 3. VERIFY PYTORCH THRESHOLD BEHAVIOR
# ======================================================================

print("\n" + "=" * 70)
print("STEP 2 — PYTORCH THRESHOLD CHECK")
print("=" * 70)

print("Original v0.9 threshold:")
print(threshold)


# Recalculate normal-event errors using PyTorch

pytorch_training_errors = []

for event in normal_events:

    error = pytorch_guardian_reconstruction_error(
        event
    )

    pytorch_training_errors.append(error)


pytorch_max_training_error = max(
    pytorch_training_errors
)

pytorch_threshold = (
    pytorch_max_training_error * 1.5
)


print("\nPyTorch maximum training error:")
print(pytorch_max_training_error)

print("\nPyTorch calculated threshold:")
print(pytorch_threshold)

print("\nDifference from stored v0.9 threshold:")
print(
    abs(
        pytorch_threshold
        - threshold
    )
)


if abs(pytorch_threshold - threshold) < 1e-5:

    print("\nPASS: PyTorch threshold agrees with v0.9.")

else:

    print("\nWARNING: Threshold difference detected.")

    print(
        "This may be due to float32 precision."
    )


# Use the original v0.9 threshold for final classification.

guardian_threshold = threshold


# ======================================================================
# 4. FULL GUARDIAN FUNCTIONS
# ======================================================================

print("\n" + "=" * 70)
print("STEP 3 — BUILDING FULL GUARDIAN")
print("=" * 70)


# ----------------------------------------------------------------------
# Context investigation
# ----------------------------------------------------------------------

def full_guardian_investigate(event):

    evidence = []

    source = event["source"]

    destination = event["destination"]


    relationship = frozenset(
        (source, destination)
    )

    direction = (
        source,
        destination
    )

    reverse_direction = (
        destination,
        source
    )

    information_key = (
        source,
        destination,
        event["information"]
    )

    protocol_key = (
        source,
        destination,
        event["protocol"],
        event["port"]
    )


    relationship_is_known = (
        relationship
        in known_relationships
    )

    direction_is_known = (
        direction
        in known_directions
    )


    # --------------------------------------------------------------
    # DEVICE CHECK
    # --------------------------------------------------------------

    if source not in devices:

        evidence.append(
            "Source device is unknown"
        )


    if destination not in devices:

        evidence.append(
            "Destination device is unknown"
        )


    # --------------------------------------------------------------
    # RELATIONSHIP + DIRECTION
    # --------------------------------------------------------------

    if not relationship_is_known:

        if reverse_direction in known_directions:

            evidence.append(
                "Known device pair communicating in a "
                "previously unseen direction"
            )

        else:

            evidence.append(
                "Previously unseen device relationship"
            )


    elif not direction_is_known:

        evidence.append(
            "Previously unseen communication direction "
            "for a known device relationship"
        )


    # --------------------------------------------------------------
    # INFORMATION CONTEXT
    # --------------------------------------------------------------

    if relationship_is_known and direction_is_known:

        if information_key not in known_information_by_direction:

            if event["information"] in known_information_types:

                evidence.append(
                    "Known information type used in a "
                    "new communication context"
                )

            else:

                evidence.append(
                    "Previously unseen information or "
                    "operation type for this communication"
                )


    # --------------------------------------------------------------
    # PROTOCOL CONTEXT
    # --------------------------------------------------------------

    if relationship_is_known and direction_is_known:

        if protocol_key not in known_protocols_by_direction:

            if (
                event["protocol"],
                event["port"]
            ) in known_protocol_types:

                evidence.append(
                    "Known protocol/port used in a "
                    "new communication context"
                )

            else:

                evidence.append(
                    "Previously unseen protocol or port "
                    "for this communication"
                )


    context = {

        "relationship_known":
            relationship_is_known,

        "direction_known":
            direction_is_known,

        "information_known_globally":
            event["information"]
            in known_information_types,

        "information_known_for_direction":
            information_key
            in known_information_by_direction,

        "protocol_known_globally":
            (
                event["protocol"],
                event["port"]
            )
            in known_protocol_types,

        "protocol_known_for_direction":
            protocol_key
            in known_protocols_by_direction
    }


    return evidence, context


# ----------------------------------------------------------------------
# Temporal investigation
# ----------------------------------------------------------------------

def full_guardian_investigate_timing(event):

    evidence = []

    if "timestamp" not in event:

        return evidence


    timestamp = event["timestamp"]

    latest_normal_timestamp = (
        normal_timestamps[-1]
    )

    time_gap = (
        timestamp
        - latest_normal_timestamp
    )


    if time_gap < 0:

        evidence.append(
            "Event timestamp is earlier than the "
            "latest observed normal timestamp"
        )

    elif time_gap > time_gap_threshold:

        evidence.append(
            "Unusual time gap between observations"
        )


    return evidence


# ----------------------------------------------------------------------
# Volume investigation
# ----------------------------------------------------------------------

def full_guardian_investigate_volume(event):

    evidence = []

    if "bytes" not in event:

        return evidence


    volume = event["bytes"]


    if volume > volume_upper_threshold:

        evidence.append(
            "Unusually large communication volume"
        )

    elif volume < volume_lower_threshold:

        evidence.append(
            "Unusually small communication volume"
        )


    return evidence


# ----------------------------------------------------------------------
# Full Guardian
# ----------------------------------------------------------------------

def full_guardian_analyze(event):

    # --------------------------------------------------------------
    # AI
    # --------------------------------------------------------------

    score = (
        pytorch_guardian_reconstruction_error(
            event
        )
    )

    ai_suspicious = (
        score > guardian_threshold
    )


    # --------------------------------------------------------------
    # Context
    # --------------------------------------------------------------

    context_evidence, context = (
        full_guardian_investigate(event)
    )


    # --------------------------------------------------------------
    # Temporal
    # --------------------------------------------------------------

    timing_evidence = (
        full_guardian_investigate_timing(
            event
        )
    )


    # --------------------------------------------------------------
    # Volume
    # --------------------------------------------------------------

    volume_evidence = (
        full_guardian_investigate_volume(
            event
        )
    )


    # --------------------------------------------------------------
    # Combine evidence
    # --------------------------------------------------------------

    evidence = []

    evidence += context_evidence
    evidence += timing_evidence
    evidence += volume_evidence


    investigation_anomaly = (
        len(evidence) > 0
    )


    # --------------------------------------------------------------
    # DECISION FUSION
    # --------------------------------------------------------------

    suspicious = (
        ai_suspicious
        or investigation_anomaly
    )


    if suspicious:

        status = "SUSPICIOUS"

    else:

        status = "NORMAL"


    return {

        "status":
            status,

        "score":
            score,

        "ai_suspicious":
            ai_suspicious,

        "investigation_anomaly":
            investigation_anomaly,

        "context_evidence":
            context_evidence,

        "timing_evidence":
            timing_evidence,

        "volume_evidence":
            volume_evidence,

        "evidence":
            evidence,

        "context":
            context
    }


print("PASS: Full Guardian functions loaded.")


# ======================================================================
# 5. HUMAN-READABLE GUARDIAN EXPLANATION
# ======================================================================

def full_guardian_explain(event, result):

    print()
    print("-" * 70)

    print("SOURCE:      ", event["source"])
    print("DESTINATION: ", event["destination"])
    print("INFORMATION: ", event["information"])
    print(
        "PROTOCOL:    ",
        event["protocol"],
        "/",
        event["port"]
    )

    if "timestamp" in event:

        print(
            "TIMESTAMP:   ",
            event["timestamp"],
            "seconds"
        )

    if "bytes" in event:

        print(
            "MESSAGE SIZE:",
            event["bytes"],
            "bytes"
        )


    print()
    print(
        "AI SCORE:    ",
        round(result["score"], 6)
    )

    print(
        "THRESHOLD:   ",
        round(guardian_threshold, 6)
    )

    print(
        "AI DECISION: ",
        "SUSPICIOUS"
        if result["ai_suspicious"]
        else "NORMAL"
    )

    print(
        "FINAL STATUS:",
        result["status"]
    )


    print()
    print("INVESTIGATION RESULTS")

    print(
        "  Context:",
        "ANOMALY FOUND"
        if result["context_evidence"]
        else "NO ANOMALY"
    )

    print(
        "  Temporal:",
        "ANOMALY FOUND"
        if result["timing_evidence"]
        else "NO ANOMALY"
    )

    print(
        "  Volume:",
        "ANOMALY FOUND"
        if result["volume_evidence"]
        else "NO ANOMALY"
    )


    print()
    print("CONTEXT")

    context = result["context"]

    print(
        "  Relationship known:",
        "YES"
        if context["relationship_known"]
        else "NO"
    )

    print(
        "  Direction known:",
        "YES"
        if context["direction_known"]
        else "NO"
    )

    print(
        "  Information known globally:",
        "YES"
        if context["information_known_globally"]
        else "NO"
    )

    print(
        "  Information known for direction:",
        "YES"
        if context["information_known_for_direction"]
        else "NO"
    )

    print(
        "  Protocol known globally:",
        "YES"
        if context["protocol_known_globally"]
        else "NO"
    )

    print(
        "  Protocol known for direction:",
        "YES"
        if context["protocol_known_for_direction"]
        else "NO"
    )


    if result["evidence"]:

        print()
        print("WHY GUARDIAN FLAGGED IT")

        for item in result["evidence"]:

            print(
                "  •",
                item
            )

    else:

        print()
        print("WHY GUARDIAN CONSIDERS IT NORMAL")

        print(
            "  • No AI, contextual, temporal, "
            "or volume anomaly detected."
        )


# ======================================================================
# 6. ORIGINAL v0.9 TEST CASES
# ======================================================================

print("\n" + "=" * 70)
print("STEP 4 — FULL GUARDIAN TEST CASES")
print("=" * 70)


guardian_tests = [

    {
        "name":
            "KNOWN NORMAL",

        "event": {
            "source":
                "PLC-01",

            "destination":
                "HIST-01",

            "information":
                "process_data",

            "protocol":
                "OPC-UA",

            "port":
                4840,

            "timestamp":
                145,

            "bytes":
                820
        },

        "expected":
            "NORMAL"
    },


    {
        "name":
            "NEW REVERSE DIRECTION",

        "event": {
            "source":
                "HIST-01",

            "destination":
                "PLC-01",

            "information":
                "process_data",

            "protocol":
                "OPC-UA",

            "port":
                4840,

            "timestamp":
                150,

            "bytes":
                820
        },

        "expected":
            "SUSPICIOUS"
    },


    {
        "name":
            "NEW INFORMATION TYPE",

        "event": {
            "source":
                "PLC-01",

            "destination":
                "HIST-01",

            "information":
                "configuration_change",

            "protocol":
                "OPC-UA",

            "port":
                4840,

            "timestamp":
                155,

            "bytes":
                820
        },

        "expected":
            "SUSPICIOUS"
    },


    {
        "name":
            "NEW PROTOCOL",

        "event": {
            "source":
                "PLC-01",

            "destination":
                "HIST-01",

            "information":
                "process_data",

            "protocol":
                "Modbus/TCP",

            "port":
                502,

            "timestamp":
                160,

            "bytes":
                820
        },

        "expected":
            "SUSPICIOUS"
    },


    {
        "name":
            "NEW RELATIONSHIP",

        "event": {
            "source":
                "PLC-02",

            "destination":
                "SIS-01",

            "information":
                "safety_status",

            "protocol":
                "Modbus/TCP",

            "port":
                502,

            "timestamp":
                165,

            "bytes":
                120
        },

        "expected":
            "SUSPICIOUS"
    },


    {
        "name":
            "KNOWN INFORMATION + NEW CONTEXT",

        "event": {
            "source":
                "PLC-02",

            "destination":
                "HMI-01",

            "information":
                "process_read",

            "protocol":
                "Modbus/TCP",

            "port":
                502,

            "timestamp":
                170,

            "bytes":
                180
        },

        "expected":
            "SUSPICIOUS"
    },


    {
        "name":
            "UNKNOWN SOURCE DEVICE",

        "event": {
            "source":
                "UNKNOWN-01",

            "destination":
                "PLC-01",

            "information":
                "process_data",

            "protocol":
                "OPC-UA",

            "port":
                4840,

            "timestamp":
                175,

            "bytes":
                820
        },

        "expected":
            "SUSPICIOUS"
    },


    {
        "name":
            "UNKNOWN DESTINATION DEVICE",

        "event": {
            "source":
                "PLC-01",

            "destination":
                "UNKNOWN-01",

            "information":
                "process_data",

            "protocol":
                "OPC-UA",

            "port":
                4840,

            "timestamp":
                180,

            "bytes":
                820
        },

        "expected":
            "SUSPICIOUS"
    },


    {
        "name":
            "MULTIPLE ANOMALIES",

        "event": {
            "source":
                "UNKNOWN-02",

            "destination":
                "UNKNOWN-03",

            "information":
                "firmware_upload",

            "protocol":
                "S7comm",

            "port":
                102,

            "timestamp":
                500,

            "bytes":
                50000
        },

        "expected":
            "SUSPICIOUS"
    },


    {
        "name":
            "UNUSUAL TIME GAP",

        "event": {
            "source":
                "PLC-01",

            "destination":
                "HIST-01",

            "information":
                "process_data",

            "protocol":
                "OPC-UA",

            "port":
                4840,

            "timestamp":
                500,

            "bytes":
                820
        },

        "expected":
            "SUSPICIOUS"
    },


    {
        "name":
            "UNUSUAL MESSAGE VOLUME",

        "event": {
            "source":
                "PLC-01",

            "destination":
                "HIST-01",

            "information":
                "process_data",

            "protocol":
                "OPC-UA",

            "port":
                4840,

            "timestamp":
                145,

            "bytes":
                5000
        },

        "expected":
            "SUSPICIOUS"
    }
]


# ======================================================================
# 7. RUN ALL TESTS
# ======================================================================

passed = 0
failed = 0

results_summary = []


for index, test in enumerate(
    guardian_tests,
    start=1
):

    print()
    print()
    print("#" * 70)

    print(
        "TEST",
        index,
        "/",
        len(guardian_tests),
        ":",
        test["name"]
    )

    print("#" * 70)


    result = full_guardian_analyze(
        test["event"]
    )


    full_guardian_explain(
        test["event"],
        result
    )


    expected = test["expected"]

    actual = result["status"]


    print()
    print("EXPECTED:", expected)
    print("ACTUAL:  ", actual)


    if actual == expected:

        print("TEST RESULT: PASS")

        passed += 1

        results_summary.append(
            (
                test["name"],
                "PASS"
            )
        )

    else:

        print("TEST RESULT: FAIL")

        failed += 1

        results_summary.append(
            (
                test["name"],
                "FAIL"
            )
        )


# ======================================================================
# 8. FINAL VALIDATION SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("FULL GUARDIAN VALIDATION SUMMARY")
print("=" * 70)


for name, result in results_summary:

    print(
        f"{result:4} : {name}"
    )


print("\n" + "-" * 70)

print(
    "Tests passed:",
    passed
)

print(
    "Tests failed:",
    failed
)


if failed == 0:

    print()
    print(
        "OVERALL RESULT:"
    )

    print(
        "ALL FULL GUARDIAN TESTS PASSED"
    )

else:

    print()
    print(
        "OVERALL RESULT:"
    )

    print(
        "SOME FULL GUARDIAN TESTS FAILED"
    )

    print(
        "Inspect the individual test output above."
    )


# ======================================================================
# 9. SAFETY BOUNDARY VERIFICATION
# ======================================================================

print("\n" + "=" * 70)
print("SAFETY BOUNDARY VERIFICATION")
print("=" * 70)

print()
print("OBSERVE     : YES")
print("ANALYZE     : YES")
print("INVESTIGATE : YES")
print("EXPLAIN     : YES")
print("ALERT       : YES")

print()

print("BLOCK TRAFFIC : NO")
print("MODIFY PLC    : NO")
print("CHANGE PROCESS: NO")
print("SHUTDOWN      : NO")
print("CONTROL OT    : NO")

print()
print(
    "PASS: Guardian remains a passive security observer."
)

print("\n" + "=" * 70)
print("FULL GUARDIAN VERIFICATION COMPLETE")
print("=" * 70)


FULL GUARDIAN VERIFICATION
PyTorch AI + Context + Temporal + Volume + Decision Fusion

STEP 1 — PYTORCH MODEL CHECK
Model:
GuardianAutoencoder(
  (encoder): Linear(in_features=18, out_features=6, bias=True)
  (decoder): Linear(in_features=6, out_features=18, bias=True)
)

Input size: 18
Expected input size: 18
PASS: Correct 18-feature input.

STEP 2 — PYTORCH THRESHOLD CHECK
Original v0.9 threshold:
0.0010467613755015663

PyTorch maximum training error:
0.0006978413439355791

PyTorch calculated threshold:
0.0010467620159033686

Difference from stored v0.9 threshold:
6.404018023341285e-10

PASS: PyTorch threshold agrees with v0.9.

STEP 3 — BUILDING FULL GUARDIAN
PASS: Full Guardian functions loaded.

STEP 4 — FULL GUARDIAN TEST CASES


######################################################################
TEST 1 / 11 : KNOWN NORMAL
######################################################################

----------------------------------------------------------------------
SOURCE:      

In [29]:
!pip install onnx

In [30]:
import torch
import onnx

print("PyTorch:", torch.__version__)
print("ONNX:", onnx.__version__)
print("Imports: PASSED")

PyTorch: 2.11.0+cpu
ONNX: 1.23.0
Imports: PASSED


In [31]:
# ==============================================================
# STEP 2 — PYTORCH → ONNX EXPORT
# ==============================================================

import torch
import onnx

print("\n" + "=" * 70)
print("STEP 2 — PYTORCH → ONNX EXPORT")
print("=" * 70)

# --------------------------------------------------------------
# 1. Inference mode
# --------------------------------------------------------------

model.eval()

# --------------------------------------------------------------
# 2. Representative input
# --------------------------------------------------------------
# Guardian Autoencoder expects 18 input features.

dummy_input = torch.randn(
    1,
    18,
    dtype=torch.float32
)

print("\nInput shape:", tuple(dummy_input.shape))

# --------------------------------------------------------------
# 3. Export
# --------------------------------------------------------------

onnx_path = "guardian_autoencoder.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["guardian_input"],
    output_names=["guardian_output"],
    dynamic_axes={
        "guardian_input": {0: "batch_size"},
        "guardian_output": {0: "batch_size"}
    },
    opset_version=17,
    dynamo=False
)

print("\nONNX export: PASSED")
print("File:", onnx_path)

# --------------------------------------------------------------
# 4. Validate ONNX structure
# --------------------------------------------------------------

onnx_model = onnx.load(onnx_path)

onnx.checker.check_model(onnx_model)

print("ONNX checker: PASSED")

# --------------------------------------------------------------
# 5. Display input/output information
# --------------------------------------------------------------

print("\nONNX INPUT")

for inp in onnx_model.graph.input:
    print(" Name :", inp.name)
    print(" Type :", inp.type.tensor_type)

print("\nONNX OUTPUT")

for out in onnx_model.graph.output:
    print(" Name :", out.name)
    print(" Type :", out.type.tensor_type)

print("\n" + "=" * 70)
print("PYTORCH → ONNX COMPLETE")
print("=" * 70)


STEP 2 — PYTORCH → ONNX EXPORT

Input shape: (1, 18)

ONNX export: PASSED
File: guardian_autoencoder.onnx
ONNX checker: PASSED

ONNX INPUT
 Name : guardian_input
 Type : elem_type: 1
shape {
  dim {
    dim_param: "batch_size"
  }
  dim {
    dim_value: 18
  }
}


ONNX OUTPUT
 Name : guardian_output
 Type : elem_type: 1
shape {
  dim {
    dim_param: "batch_size"
  }
  dim {
    dim_value: 18
  }
}


PYTORCH → ONNX COMPLETE


/tmp/ipykernel_15903/3317661262.py:37: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [32]:
print(type(model))
!pip install onnxruntime

<class '__main__.GuardianAutoencoder'>


In [33]:
import onnxruntime as ort

print("ONNX Runtime:", ort.__version__)
print("ONNX Runtime import: PASSED")

ONNX Runtime: 1.30.0
ONNX Runtime import: PASSED


In [34]:
# ==============================================================
# STEP 3 — PYTORCH vs ONNX NUMERICAL VERIFICATION
# ==============================================================

import numpy as np
import torch
import onnxruntime as ort

print("\n" + "=" * 70)
print("STEP 3 — PYTORCH vs ONNX NUMERICAL VERIFICATION")
print("=" * 70)

# --------------------------------------------------------------
# 1. Put PyTorch model in inference mode
# --------------------------------------------------------------

model.eval()

# --------------------------------------------------------------
# 2. Create deterministic test input
# --------------------------------------------------------------
# 5 observations × 18 Guardian features

torch.manual_seed(42)

test_input = torch.randn(
    5,
    18,
    dtype=torch.float32
)

# --------------------------------------------------------------
# 3. PyTorch inference
# --------------------------------------------------------------

with torch.no_grad():
    pytorch_output = model(test_input)

pytorch_output = pytorch_output.cpu().numpy()

# --------------------------------------------------------------
# 4. Load ONNX model
# --------------------------------------------------------------

onnx_path = "guardian_autoencoder.onnx"

session = ort.InferenceSession(
    onnx_path,
    providers=["CPUExecutionProvider"]
)

# --------------------------------------------------------------
# 5. ONNX inference
# --------------------------------------------------------------

onnx_output = session.run(
    ["guardian_output"],
    {
        "guardian_input": test_input.cpu().numpy()
    }
)[0]

# --------------------------------------------------------------
# 6. Numerical comparison
# --------------------------------------------------------------

absolute_difference = np.abs(
    pytorch_output - onnx_output
)

max_difference = np.max(
    absolute_difference
)

mean_difference = np.mean(
    absolute_difference
)

# --------------------------------------------------------------
# 7. np.allclose verification
# --------------------------------------------------------------

numerically_close = np.allclose(
    pytorch_output,
    onnx_output,
    rtol=1e-5,
    atol=1e-6
)

# --------------------------------------------------------------
# 8. Results
# --------------------------------------------------------------

print("\nTest input shape:")
print(" ", test_input.shape)

print("\nPyTorch output shape:")
print(" ", pytorch_output.shape)

print("\nONNX output shape:")
print(" ", onnx_output.shape)

print("\nMaximum absolute difference:")
print(f"  {max_difference:.10f}")

print("\nMean absolute difference:")
print(f"  {mean_difference:.10f}")

print("\nNumerical equivalence:")
print(
    "  PASSED"
    if numerically_close
    else "  FAILED"
)

print("\n" + "=" * 70)

if numerically_close:
    print("PYTORCH ↔ ONNX NUMERICAL VERIFICATION: PASSED")
else:
    print("PYTORCH ↔ ONNX NUMERICAL VERIFICATION: FAILED")

print("=" * 70)


STEP 3 — PYTORCH vs ONNX NUMERICAL VERIFICATION

Test input shape:
  torch.Size([5, 18])

PyTorch output shape:
  (5, 18)

ONNX output shape:
  (5, 18)

Maximum absolute difference:
  0.0000003278

Mean absolute difference:
  0.0000000517

Numerical equivalence:
  PASSED

PYTORCH ↔ ONNX NUMERICAL VERIFICATION: PASSED


In [51]:
# ==============================================================
# STEP 6 — STATIC ONNX EXPORT FOR QUALCOMM AI HUB
# ==============================================================

import torch
import onnx

print("\n" + "=" * 70)
print("STEP 6 — STATIC ONNX EXPORT FOR QUALCOMM AI HUB")
print("=" * 70)

model.eval()

# --------------------------------------------------------------
# Static deployment input
# --------------------------------------------------------------
# One Guardian observation containing 18 features.

static_input = torch.randn(
    1,
    18,
    dtype=torch.float32
)

static_onnx_path = "guardian_autoencoder_static.onnx"

# --------------------------------------------------------------
# Export WITHOUT dynamic axes
# --------------------------------------------------------------

torch.onnx.export(
    model,
    static_input,
    static_onnx_path,

    input_names=["guardian_input"],
    output_names=["guardian_output"],

    opset_version=17,

    dynamo=False
)

print("\nStatic ONNX export: PASSED")
print("File:", static_onnx_path)

# --------------------------------------------------------------
# Validate
# --------------------------------------------------------------

static_onnx_model = onnx.load(static_onnx_path)

onnx.checker.check_model(static_onnx_model)

print("ONNX checker: PASSED")

# --------------------------------------------------------------
# Display shapes
# --------------------------------------------------------------

print("\nINPUT SHAPE")

for inp in static_onnx_model.graph.input:
    shape = inp.type.tensor_type.shape

    dims = []

    for dim in shape.dim:
        if dim.dim_value:
            dims.append(dim.dim_value)
        elif dim.dim_param:
            dims.append(dim.dim_param)

    print(f"  {inp.name}: {dims}")

print("\nOUTPUT SHAPE")

for out in static_onnx_model.graph.output:
    shape = out.type.tensor_type.shape

    dims = []

    for dim in shape.dim:
        if dim.dim_value:
            dims.append(dim.dim_value)
        elif dim.dim_param:
            dims.append(dim.dim_param)

    print(f"  {out.name}: {dims}")

print("\n" + "=" * 70)
print("STATIC ONNX MODEL READY")
print("=" * 70)


STEP 6 — STATIC ONNX EXPORT FOR QUALCOMM AI HUB

Static ONNX export: PASSED
File: guardian_autoencoder_static.onnx
ONNX checker: PASSED

INPUT SHAPE
  guardian_input: [1, 18]

OUTPUT SHAPE
  guardian_output: [1, 18]

STATIC ONNX MODEL READY


/tmp/ipykernel_15903/1650500262.py:31: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [52]:
# ==============================================================
# STEP 6A — STATIC ONNX NUMERICAL VERIFICATION
# ==============================================================

import numpy as np
import torch
import onnxruntime as ort

print("\n" + "=" * 70)
print("STEP 6A — STATIC ONNX NUMERICAL VERIFICATION")
print("=" * 70)

model.eval()

# --------------------------------------------------------------
# Deterministic test input
# --------------------------------------------------------------

torch.manual_seed(42)

test_input = torch.randn(
    1,
    18,
    dtype=torch.float32
)

# --------------------------------------------------------------
# PyTorch inference
# --------------------------------------------------------------

with torch.no_grad():
    pytorch_output = model(test_input)

pytorch_output = pytorch_output.cpu().numpy()

# --------------------------------------------------------------
# Static ONNX inference
# --------------------------------------------------------------

session = ort.InferenceSession(
    "guardian_autoencoder_static.onnx",
    providers=["CPUExecutionProvider"]
)

onnx_output = session.run(
    ["guardian_output"],
    {
        "guardian_input": test_input.cpu().numpy()
    }
)[0]

# --------------------------------------------------------------
# Compare
# --------------------------------------------------------------

absolute_difference = np.abs(
    pytorch_output - onnx_output
)

max_difference = np.max(
    absolute_difference
)

mean_difference = np.mean(
    absolute_difference
)

numerically_close = np.allclose(
    pytorch_output,
    onnx_output,
    rtol=1e-5,
    atol=1e-6
)

# --------------------------------------------------------------
# Results
# --------------------------------------------------------------

print("\nInput shape:")
print(" ", test_input.shape)

print("\nPyTorch output shape:")
print(" ", pytorch_output.shape)

print("\nONNX output shape:")
print(" ", onnx_output.shape)

print("\nMaximum absolute difference:")
print(f"  {max_difference:.10f}")

print("\nMean absolute difference:")
print(f"  {mean_difference:.10f}")

print("\nNumerical equivalence:")
print(
    "  PASSED"
    if numerically_close
    else "  FAILED"
)

print("\n" + "=" * 70)

if numerically_close:
    print("STATIC PYTORCH ↔ ONNX VERIFICATION: PASSED")
else:
    print("STATIC PYTORCH ↔ ONNX VERIFICATION: FAILED")

print("=" * 70)


STEP 6A — STATIC ONNX NUMERICAL VERIFICATION

Input shape:
  torch.Size([1, 18])

PyTorch output shape:
  (1, 18)

ONNX output shape:
  (1, 18)

Maximum absolute difference:
  0.0000000894

Mean absolute difference:
  0.0000000351

Numerical equivalence:
  PASSED

STATIC PYTORCH ↔ ONNX VERIFICATION: PASSED


In [35]:
!pip install -U qai-hub

In [40]:
import qai_hub as hub

print("QAI Hub version:", hub.__version__)
print("QAI Hub import: PASSED")
client = hub.Client()

client_config = hub.ClientConfig(
    api_token=""
)

client = hub.Client(client_config)

print("Qualcomm AI Hub client: CONNECTED")
devices = client.get_devices()

print("AI Hub connection: PASSED")
print("Available devices:", len(devices))

QAI Hub version: 0.55.0
QAI Hub import: PASSED
Qualcomm AI Hub client: CONNECTED
AI Hub connection: PASSED
Available devices: 78


In [41]:
# ==============================================================
# STEP 4 — IDENTIFY SNAPDRAGON X ELITE TARGET
# ==============================================================

x_elite_devices = client.get_devices(
    name="Snapdragon X Elite CRD"
)

print("\n" + "=" * 70)
print("SNAPDRAGON X ELITE DEVICE CHECK")
print("=" * 70)

print("\nMatching devices:", len(x_elite_devices))

for device in x_elite_devices:
    print("\nDevice:")
    print(device)

print("\n" + "=" * 70)


SNAPDRAGON X ELITE DEVICE CHECK

Matching devices: 1

Device:
Device(name='Snapdragon X Elite CRD', os='11', attributes=['os:windows', 'format:compute', 'framework:onnx', 'vendor:qualcomm', 'abi:aarch64-windows', 'chipset:qualcomm-snapdragon-x-elite', 'chipset:sc8380xp', 'hexagon:v73', 'soc-model:60', 'htp-supports-fp16:true', 'htp-supports-weight-sharing:true', 'framework:qnn'])



In [42]:
print([x for x in dir(client) if "compile" in x.lower()])
print([x for x in dir(hub) if "compile" in x.lower()])

['submit_compile_and_link_jobs', 'submit_compile_and_profile_jobs', 'submit_compile_job']
['CompileJob', 'CompileJobResult', 'CompileJobSummary', 'submit_compile_and_link_jobs', 'submit_compile_and_profile_jobs', 'submit_compile_job']


In [53]:
# ==============================================================
# STEP 7 — QUALCOMM AI HUB COMPILE
# ==============================================================

import qai_hub as hub

print("\n" + "=" * 70)
print("STEP 7 — QUALCOMM AI HUB COMPILE")
print("=" * 70)

target_device = hub.Device("Snapdragon X Elite CRD")

compile_job = client.submit_compile_job(
    model="guardian_autoencoder_static.onnx",
    device=target_device
)

print("\nCompile job submitted.")
print("Job ID:")
print(" ", compile_job.job_id)

print("\nTarget device:")
print(" ", compile_job.device)

print("\nInitial status:")
print(" ", compile_job.get_status())

print("\n" + "=" * 70)
print("STEP 7 SUBMISSION: PASSED")
print("=" * 70)


STEP 7 — QUALCOMM AI HUB COMPILE

Compile job submitted.
Job ID:
  jglyxeel5

Target device:
  Device(name='Snapdragon X Elite CRD', os='11', attributes=['os:windows', 'format:compute', 'framework:onnx', 'vendor:qualcomm', 'abi:aarch64-windows', 'chipset:qualcomm-snapdragon-x-elite', 'chipset:sc8380xp', 'hexagon:v73', 'soc-model:60', 'htp-supports-fp16:true', 'htp-supports-weight-sharing:true', 'framework:qnn'])

Initial status:
  JobStatus
---------
code    : CREATED
message : 


STEP 7 SUBMISSION: PASSED


In [54]:
# ==============================================================
# STEP 7A — WAIT FOR QUALCOMM AI HUB COMPILE
# ==============================================================

print("\n" + "=" * 70)
print("STEP 7A — WAITING FOR QUALCOMM AI HUB")
print("=" * 70)

final_status = compile_job.wait()

print("\nFinal compile status:")
print(final_status)

print("\n" + "=" * 70)

if final_status.code == "DONE":
    print("QUALCOMM AI HUB COMPILE: PASSED")
else:
    print("QUALCOMM AI HUB COMPILE RESULT:")
    print(final_status.code)
    print(final_status.message)

print("=" * 70)


STEP 7A — WAITING FOR QUALCOMM AI HUB

Final compile status:
JobStatus
---------
code    : SUCCESS
message : 


QUALCOMM AI HUB COMPILE RESULT:
SUCCESS



In [55]:
# ==============================================================
# STEP 8 — RETRIEVE QUALCOMM COMPILED MODEL
# ==============================================================

print("\n" + "=" * 70)
print("STEP 8 — QUALCOMM COMPILED MODEL")
print("=" * 70)

print("\nAvailable artifacts:")
print(compile_job.get_available_artifacts())

print("\nTarget model information:")
print(compile_job.get_target_model())

print("\n" + "=" * 70)


STEP 8 — QUALCOMM COMPILED MODEL

Available artifacts:
[<JobArtifactType.HUB_LOG: 5>]

Target model information:
Model(model_id='mqkp1rrzm', name='future-model-jglyxeel5')



In [56]:
# ==============================================================
# STEP 9 — QUALCOMM AI HUB PROFILING
# ==============================================================

print("\n" + "=" * 70)
print("STEP 9 — QUALCOMM AI HUB PROFILING")
print("=" * 70)

profile_job = client.submit_profile_job(
    model=compile_job.get_target_model(),
    device=target_device
)

print("\nProfile job submitted.")
print("Job ID:")
print(" ", profile_job.job_id)

print("\nTarget device:")
print(" ", profile_job.device)

print("\nInitial status:")
print(" ", profile_job.get_status())

print("\n" + "=" * 70)
print("STEP 9 SUBMISSION: PASSED")
print("=" * 70)


STEP 9 — QUALCOMM AI HUB PROFILING

Profile job submitted.
Job ID:
  jg9zm99vp

Target device:
  Device(name='Snapdragon X Elite CRD', os='11', attributes=['os:windows', 'format:compute', 'framework:onnx', 'vendor:qualcomm', 'abi:aarch64-windows', 'chipset:qualcomm-snapdragon-x-elite', 'chipset:sc8380xp', 'hexagon:v73', 'soc-model:60', 'htp-supports-fp16:true', 'htp-supports-weight-sharing:true', 'framework:qnn'])

Initial status:
  JobStatus
---------
code    : CREATED
message : 


STEP 9 SUBMISSION: PASSED


In [57]:
# ==============================================================
# STEP 9A — WAIT FOR QUALCOMM AI HUB PROFILE
# ==============================================================

print("\n" + "=" * 70)
print("STEP 9A — WAITING FOR QUALCOMM AI HUB PROFILE")
print("=" * 70)

profile_status = profile_job.wait()

print("\nFinal profile status:")
print(profile_status)

print("\n" + "=" * 70)

if profile_status.code == "SUCCESS":
    print("QUALCOMM AI HUB PROFILE: SUCCESS")
else:
    print("QUALCOMM AI HUB PROFILE RESULT:")
    print(profile_status.code)
    print(profile_status.message)

print("=" * 70)


STEP 9A — WAITING FOR QUALCOMM AI HUB PROFILE

Final profile status:
JobStatus
---------
code    : SUCCESS
message : 


QUALCOMM AI HUB PROFILE: SUCCESS


In [59]:
# ==============================================================
# STEP 10A — INSPECT PROFILE JOB API
# ==============================================================

print("\n" + "=" * 70)
print("STEP 10A — PROFILE JOB API")
print("=" * 70)

print("\nProfile job methods:")
print([
    x for x in dir(profile_job)
    if not x.startswith("_")
])

print("\nAvailable artifacts:")
print(profile_job.get_available_artifacts())

print("\n" + "=" * 70)


STEP 10A — PROFILE JOB API

Profile job methods:
['date', 'device', 'disable_sharing', 'download_artifacts_for_type', 'download_job_logs', 'download_profile', 'download_results', 'get_available_artifacts', 'get_sharing', 'get_status', 'hub_version', 'job_id', 'job_type', 'model', 'modify_sharing', 'name', 'options', 'project_id', 'set_name', 'shapes', 'url', 'verbose', 'wait']

Available artifacts:
[<JobArtifactType.DEVICE_LOG: 1>, <JobArtifactType.PROFILE_TRACE: 6>, <JobArtifactType.QHAS_SUMMARY: 8>]



In [60]:
# ==============================================================
# STEP 10B — DOWNLOAD QUALCOMM PROFILE
# ==============================================================

import os

print("\n" + "=" * 70)
print("STEP 10B — QUALCOMM PROFILE")
print("=" * 70)

profile_dir = "guardian_profile"

os.makedirs(profile_dir, exist_ok=True)

profile_data = profile_job.download_profile(
    profile_dir
)

print("\nProfile downloaded.")
print("Location:")
print(" ", profile_dir)

print("\nReturned value:")
print(profile_data)

print("\n" + "=" * 70)
print("STEP 10B — PROFILE RETRIEVAL COMPLETE")
print("=" * 70)


STEP 10B — QUALCOMM PROFILE
Saved profile results to guardian_profile/future-model-jglyxeel5_jg9zm99vp_results.json

Profile downloaded.
Location:
  guardian_profile

Returned value:
guardian_profile/future-model-jglyxeel5_jg9zm99vp_results.json

STEP 10B — PROFILE RETRIEVAL COMPLETE


In [61]:
# ==============================================================
# STEP 11 — INSPECT QUALCOMM PROFILE RESULTS
# ==============================================================

import json

print("\n" + "=" * 70)
print("STEP 11 — QUALCOMM PROFILE RESULTS")
print("=" * 70)

profile_path = "guardian_profile/future-model-jglyxeel5_jg9zm99vp_results.json"

with open(profile_path, "r") as f:
    profile_data = json.load(f)

print("\nTop-level result type:")
print(" ", type(profile_data).__name__)

print("\nProfile contents:")
print(json.dumps(profile_data, indent=2))

print("\n" + "=" * 70)
print("STEP 11 — PROFILE INSPECTION COMPLETE")
print("=" * 70)


STEP 11 — QUALCOMM PROFILE RESULTS

Top-level result type:
  dict

Profile contents:
{
  "execution_summary": {
    "estimated_inference_time": 31,
    "estimated_inference_peak_memory": 28749824,
    "first_load_time": 1798056,
    "first_load_peak_memory": 275017728,
    "warm_load_time": 567347,
    "warm_load_peak_memory": 28618752,
    "compile_time": 0,
    "compile_peak_memory": 0,
    "compile_memory_increase_range": [
      0,
      0
    ],
    "compile_memory_peak_range": [
      0,
      0
    ],
    "first_load_memory_increase_range": [
      237817856,
      237817856
    ],
    "first_load_memory_peak_range": [
      270143488,
      270143488
    ],
    "warm_load_memory_increase_range": [
      23736320,
      23736320
    ],
    "warm_load_memory_peak_range": [
      23736320,
      23736320
    ],
    "inference_memory_increase_range": [
      1060864,
      1060864
    ],
    "inference_memory_peak_range": [
      1060864,
      1060864
    ],
    "all_compile_time

In [62]:
# ==============================================================
# STEP 12 — QUALCOMM TARGET MODEL INFERENCE
# ==============================================================

print("\n" + "=" * 70)
print("STEP 12 — QUALCOMM TARGET MODEL INFERENCE")
print("=" * 70)

# Reuse the Qualcomm target model generated by the successful
# compile job.
target_model = compile_job.get_target_model()

# Use the same deterministic test input used for ONNX verification.
torch.manual_seed(42)

test_input = torch.randn(
    1,
    18,
    dtype=torch.float32
)

# Submit inference to the Snapdragon X Elite target.
inference_job = client.submit_inference_job(
    model=target_model,
    device=target_device,
    inputs={
        "guardian_input": [test_input.numpy()]
    }
)

print("\nInference job submitted.")
print("Job ID:")
print(" ", inference_job.job_id)

print("\nTarget device:")
print(" ", inference_job.device)

print("\nInitial status:")
print(" ", inference_job.get_status())

print("\n" + "=" * 70)
print("STEP 12 SUBMISSION COMPLETE")
print("=" * 70)


STEP 12 — QUALCOMM TARGET MODEL INFERENCE

Inference job submitted.
Job ID:
  jpxl4ex1p

Target device:
  Device(name='Snapdragon X Elite CRD', os='11', attributes=['os:windows', 'format:compute', 'framework:onnx', 'vendor:qualcomm', 'abi:aarch64-windows', 'chipset:qualcomm-snapdragon-x-elite', 'chipset:sc8380xp', 'hexagon:v73', 'soc-model:60', 'htp-supports-fp16:true', 'htp-supports-weight-sharing:true', 'framework:qnn'])

Initial status:
  JobStatus
---------
code    : CREATED
message : 


STEP 12 SUBMISSION COMPLETE


In [63]:
# ==============================================================
# STEP 12A — WAIT FOR QUALCOMM INFERENCE
# ==============================================================

print("\n" + "=" * 70)
print("STEP 12A — WAITING FOR QUALCOMM INFERENCE")
print("=" * 70)

inference_status = inference_job.wait()

print("\nFinal inference status:")
print(inference_status)

print("\n" + "=" * 70)

if inference_status.code == "SUCCESS":
    print("QUALCOMM AI HUB INFERENCE: SUCCESS")
else:
    print("QUALCOMM AI HUB INFERENCE RESULT:")
    print(inference_status.code)
    print(inference_status.message)

print("=" * 70)


STEP 12A — WAITING FOR QUALCOMM INFERENCE

Final inference status:
JobStatus
---------
code    : SUCCESS
message : 


QUALCOMM AI HUB INFERENCE: SUCCESS


In [64]:
# ==============================================================
# STEP 13 — RETRIEVE QUALCOMM INFERENCE OUTPUT
# ==============================================================

print("\n" + "=" * 70)
print("STEP 13 — QUALCOMM INFERENCE OUTPUT")
print("=" * 70)

inference_dir = "guardian_inference"

import os
os.makedirs(inference_dir, exist_ok=True)

inference_results = inference_job.download_results(
    inference_dir
)

print("\nInference results downloaded.")
print("Location:")
print(" ", inference_dir)

print("\nReturned value:")
print(inference_results)

print("\n" + "=" * 70)
print("STEP 13 — INFERENCE OUTPUT RETRIEVED")
print("=" * 70)


STEP 13 — QUALCOMM INFERENCE OUTPUT
Downloaded dataset to /content/guardian_inference/dataset-d930odq39.h5

Inference results downloaded.
Location:
  guardian_inference

Returned value:
InferenceJobResult
------------------
status        : JobStatus
---------
code    : SUCCESS
message : 

artifacts_dir : /content/guardian_inference


STEP 13 — INFERENCE OUTPUT RETRIEVED


In [65]:
# ==============================================================
# STEP 14 — INSPECT QUALCOMM INFERENCE DATASET
# ==============================================================

import h5py

print("\n" + "=" * 70)
print("STEP 14 — QUALCOMM INFERENCE DATASET")
print("=" * 70)

inference_file = "/content/guardian_inference/dataset-d930odq39.h5"

with h5py.File(inference_file, "r") as f:

    print("\nTop-level keys:")
    for key in f.keys():
        print(" ", key)

    print("\nDataset structure:")

    def show_structure(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(
                f"  {name} | "
                f"shape={obj.shape} | "
                f"dtype={obj.dtype}"
            )
        elif isinstance(obj, h5py.Group):
            print(f"  {name}/")

    f.visititems(show_structure)

print("\n" + "=" * 70)
print("STEP 14 — DATASET INSPECTION COMPLETE")
print("=" * 70)


STEP 14 — QUALCOMM INFERENCE DATASET

Top-level keys:
  data

Dataset structure:
  data/
  data/0/
  data/0/batch_0 | shape=(1, 18) | dtype=float32

STEP 14 — DATASET INSPECTION COMPLETE


In [66]:
# ==============================================================
# STEP 15 — PYTORCH ↔ ONNX ↔ QUALCOMM OUTPUT VERIFICATION
# ==============================================================

import numpy as np
import torch
import onnxruntime as ort
import h5py

print("\n" + "=" * 70)
print("STEP 15 — PYTORCH ↔ ONNX ↔ QUALCOMM VERIFICATION")
print("=" * 70)

# --------------------------------------------------------------
# 1. Recreate the exact deterministic input
# --------------------------------------------------------------

model.eval()

torch.manual_seed(42)

test_input = torch.randn(
    1,
    18,
    dtype=torch.float32
)

# --------------------------------------------------------------
# 2. PyTorch output
# --------------------------------------------------------------

with torch.no_grad():
    pytorch_output = model(test_input)

pytorch_output = pytorch_output.cpu().numpy()

# --------------------------------------------------------------
# 3. ONNX output
# --------------------------------------------------------------

onnx_session = ort.InferenceSession(
    "guardian_autoencoder_static.onnx",
    providers=["CPUExecutionProvider"]
)

onnx_output = onnx_session.run(
    ["guardian_output"],
    {
        "guardian_input": test_input.cpu().numpy()
    }
)[0]

# --------------------------------------------------------------
# 4. Qualcomm inference output
# --------------------------------------------------------------

inference_file = (
    "/content/guardian_inference/"
    "dataset-d930odq39.h5"
)

with h5py.File(inference_file, "r") as f:
    qualcomm_output = np.array(
        f["data/0/batch_0"]
    )

# --------------------------------------------------------------
# 5. Compare PyTorch ↔ ONNX
# --------------------------------------------------------------

pytorch_onnx_diff = np.abs(
    pytorch_output - onnx_output
)

pytorch_onnx_max = np.max(
    pytorch_onnx_diff
)

pytorch_onnx_mean = np.mean(
    pytorch_onnx_diff
)

pytorch_onnx_close = np.allclose(
    pytorch_output,
    onnx_output,
    rtol=1e-5,
    atol=1e-6
)

# --------------------------------------------------------------
# 6. Compare PyTorch ↔ Qualcomm
# --------------------------------------------------------------

pytorch_qualcomm_diff = np.abs(
    pytorch_output - qualcomm_output
)

pytorch_qualcomm_max = np.max(
    pytorch_qualcomm_diff
)

pytorch_qualcomm_mean = np.mean(
    pytorch_qualcomm_diff
)

pytorch_qualcomm_close = np.allclose(
    pytorch_output,
    qualcomm_output,
    rtol=1e-5,
    atol=1e-6
)

# --------------------------------------------------------------
# 7. Compare ONNX ↔ Qualcomm
# --------------------------------------------------------------

onnx_qualcomm_diff = np.abs(
    onnx_output - qualcomm_output
)

onnx_qualcomm_max = np.max(
    onnx_qualcomm_diff
)

onnx_qualcomm_mean = np.mean(
    onnx_qualcomm_diff
)

onnx_qualcomm_close = np.allclose(
    onnx_output,
    qualcomm_output,
    rtol=1e-5,
    atol=1e-6
)

# --------------------------------------------------------------
# 8. Report
# --------------------------------------------------------------

print("\nInput shape:")
print(" ", test_input.shape)

print("\nPyTorch output:")
print(" ", pytorch_output.shape)

print("\nONNX output:")
print(" ", onnx_output.shape)

print("\nQualcomm output:")
print(" ", qualcomm_output.shape)

print("\n" + "-" * 70)

print("\nPyTorch ↔ ONNX")
print("  Maximum difference:", f"{pytorch_onnx_max:.10f}")
print("  Mean difference:   ", f"{pytorch_onnx_mean:.10f}")
print("  Result:             ",
      "PASSED" if pytorch_onnx_close else "FAILED")

print("\nPyTorch ↔ Qualcomm")
print("  Maximum difference:", f"{pytorch_qualcomm_max:.10f}")
print("  Mean difference:   ", f"{pytorch_qualcomm_mean:.10f}")
print("  Result:             ",
      "PASSED" if pytorch_qualcomm_close else "FAILED")

print("\nONNX ↔ Qualcomm")
print("  Maximum difference:", f"{onnx_qualcomm_max:.10f}")
print("  Mean difference:   ", f"{onnx_qualcomm_mean:.10f}")
print("  Result:             ",
      "PASSED" if onnx_qualcomm_close else "FAILED")

print("\n" + "=" * 70)

if (
    pytorch_onnx_close
    and pytorch_qualcomm_close
    and onnx_qualcomm_close
):
    print("PYTORCH ↔ ONNX ↔ QUALCOMM VERIFICATION: PASSED")
else:
    print("PYTORCH ↔ ONNX ↔ QUALCOMM VERIFICATION: FAILED")

print("=" * 70)


STEP 15 — PYTORCH ↔ ONNX ↔ QUALCOMM VERIFICATION

Input shape:
  torch.Size([1, 18])

PyTorch output:
  (1, 18)

ONNX output:
  (1, 18)

Qualcomm output:
  (1, 18)

----------------------------------------------------------------------

PyTorch ↔ ONNX
  Maximum difference: 0.0000000894
  Mean difference:    0.0000000351
  Result:              PASSED

PyTorch ↔ Qualcomm
  Maximum difference: 0.0080730021
  Mean difference:    0.0021878735
  Result:              FAILED

ONNX ↔ Qualcomm
  Maximum difference: 0.0080729872
  Mean difference:    0.0021878548
  Result:              FAILED

PYTORCH ↔ ONNX ↔ QUALCOMM VERIFICATION: FAILED


In [67]:
# ==============================================================
# STEP 16 — DETAILED QUALCOMM OUTPUT INSPECTION
# ==============================================================

print("\n" + "=" * 70)
print("STEP 16 — DETAILED QUALCOMM OUTPUT INSPECTION")
print("=" * 70)

np.set_printoptions(
    precision=8,
    suppress=False
)

print("\nInput:")
print(test_input.numpy())

print("\nPyTorch output:")
print(pytorch_output)

print("\nONNX output:")
print(onnx_output)

print("\nQualcomm output:")
print(qualcomm_output)

print("\n" + "-" * 70)

print("\nPyTorch - Qualcomm:")
print(pytorch_output - qualcomm_output)

print("\nAbsolute difference:")
print(np.abs(pytorch_output - qualcomm_output))

print("\n" + "=" * 70)


STEP 16 — DETAILED QUALCOMM OUTPUT INSPECTION

Input:
[[ 1.9269153   1.4872841  -0.49739754  0.43958926 -0.75813115  1.0783176
   0.80080056  1.6806206   0.3558599  -0.686623    0.6104665   1.3347378
  -0.23162432  0.04175949 -0.2515753   0.8598585  -0.30972677 -0.3957105 ]]

PyTorch output:
[[1.96089551e-01 8.22408795e-02 9.16893959e-01 1.12647002e-04
  2.11938891e-06 2.06315577e-01 8.33676398e-01 1.53426036e-01
  3.70690003e-02 5.23537835e-07 1.94390297e-01 2.01633275e-01
  2.01601431e-01 5.84311457e-03 2.07085827e-06 1.11055066e-04
  7.85675347e-01 2.12548509e-01]]

ONNX output:
[[1.96089566e-01 8.22408199e-02 9.16893959e-01 1.12652779e-04
  2.08616257e-06 2.06315666e-01 8.33676338e-01 1.53426051e-01
  3.70690227e-02 5.36441803e-07 1.94390237e-01 2.01633245e-01
  2.01601475e-01 5.84313273e-03 2.05636024e-06 1.11073256e-04
  7.85675287e-01 2.12548584e-01]]

Qualcomm output:
[[0.20178224 0.08050538 0.9179688  0.         0.         0.20703126
  0.83105475 0.16149904 0.03399659 0.     

In [69]:
# ==============================================================
# STEP 17A — INSPECT GUARDIAN AUTOENCODER CONSTRUCTOR
# ==============================================================

import inspect

print("\n" + "=" * 70)
print("STEP 17A — GUARDIAN AUTOENCODER INSPECTION")
print("=" * 70)

print("\nConstructor signature:")
print(inspect.signature(GuardianAutoencoder))

print("\nCurrent model:")
print(model)

print("\n" + "=" * 70)


STEP 17A — GUARDIAN AUTOENCODER INSPECTION

Constructor signature:
()

Current model:
GuardianAutoencoder(
  (encoder): Linear(in_features=18, out_features=6, bias=True)
  (decoder): Linear(in_features=6, out_features=18, bias=True)
)



In [70]:
# ==============================================================
# STEP 17B — FP16 NUMERICAL DIAGNOSTIC
# ==============================================================

print("\n" + "=" * 70)
print("STEP 17B — FP16 NUMERICAL DIAGNOSTIC")
print("=" * 70)

# Create a separate model using the actual constructor.
fp16_model = GuardianAutoencoder()

# Copy the exact trained weights.
fp16_model.load_state_dict(model.state_dict())

# Convert ONLY this diagnostic copy to FP16.
fp16_model = fp16_model.half()
fp16_model.eval()

# Same deterministic input, converted to FP16.
fp16_input = test_input.half()

with torch.no_grad():
    fp16_output = fp16_model(fp16_input)

# Convert result back to FP32 for comparison.
fp16_output = fp16_output.float().cpu().numpy()

# Compare FP16 PyTorch against Qualcomm.
fp16_qualcomm_diff = np.abs(
    fp16_output - qualcomm_output
)

fp16_qualcomm_max = np.max(
    fp16_qualcomm_diff
)

fp16_qualcomm_mean = np.mean(
    fp16_qualcomm_diff
)

print("\nFP16 PyTorch output:")
print(fp16_output)

print("\nQualcomm output:")
print(qualcomm_output)

print("\n" + "-" * 70)

print("\nFP16 PyTorch ↔ Qualcomm")
print("  Maximum difference:",
      f"{fp16_qualcomm_max:.10f}")

print("  Mean difference:   ",
      f"{fp16_qualcomm_mean:.10f}")

print("\nOriginal FP32 PyTorch ↔ Qualcomm")
print("  Maximum difference:",
      f"{pytorch_qualcomm_max:.10f}")

print("  Mean difference:   ",
      f"{pytorch_qualcomm_mean:.10f}")

print("\n" + "=" * 70)
print("STEP 17B — DIAGNOSTIC COMPLETE")
print("=" * 70)


STEP 17B — FP16 NUMERICAL DIAGNOSTIC

FP16 PyTorch output:
[[1.96044922e-01 8.23974609e-02 9.16992188e-01 1.13248825e-04
  2.14576721e-06 2.06054688e-01 8.33496094e-01 1.53442383e-01
  3.70483398e-02 5.36441803e-07 1.94213867e-01 2.01293945e-01
  2.01538086e-01 5.86700439e-03 2.08616257e-06 1.10626221e-04
  7.86132812e-01 2.12524414e-01]]

Qualcomm output:
[[0.20178224 0.08050538 0.9179688  0.         0.         0.20703126
  0.83105475 0.16149904 0.03399659 0.         0.1994629  0.20251466
  0.2072754  0.00866699 0.         0.         0.784668   0.21325685]]

----------------------------------------------------------------------

FP16 PyTorch ↔ Qualcomm
  Maximum difference: 0.0080566555
  Mean difference:    0.0022536269

Original FP32 PyTorch ↔ Qualcomm
  Maximum difference: 0.0080730021
  Mean difference:    0.0021878735

STEP 17B — DIAGNOSTIC COMPLETE


In [71]:
# ==============================================================
# STEP 18A — RECONSTRUCTION ERROR COMPARISON
# ==============================================================

print("\n" + "=" * 70)
print("STEP 18A — RECONSTRUCTION ERROR COMPARISON")
print("=" * 70)

# Same deterministic input
x = test_input.numpy()

# PyTorch reconstruction error
pytorch_error = np.mean(
    np.square(x - pytorch_output),
    axis=1
)

# ONNX reconstruction error
onnx_error = np.mean(
    np.square(x - onnx_output),
    axis=1
)

# Qualcomm reconstruction error
qualcomm_error = np.mean(
    np.square(x - qualcomm_output),
    axis=1
)

print("\nReconstruction errors:")
print("  PyTorch :", pytorch_error)
print("  ONNX    :", onnx_error)
print("  Qualcomm:", qualcomm_error)

print("\n" + "-" * 70)

print("PyTorch ↔ ONNX error difference:",
      np.abs(pytorch_error - onnx_error))

print("PyTorch ↔ Qualcomm error difference:",
      np.abs(pytorch_error - qualcomm_error))

print("ONNX ↔ Qualcomm error difference:",
      np.abs(onnx_error - qualcomm_error))

print("\n" + "=" * 70)
print("STEP 18A — COMPLETE")
print("=" * 70)


STEP 18A — RECONSTRUCTION ERROR COMPARISON

Reconstruction errors:
  PyTorch : [0.8568479]
  ONNX    : [0.8568479]
  Qualcomm: [0.85472083]

----------------------------------------------------------------------
PyTorch ↔ ONNX error difference: [0.]
PyTorch ↔ Qualcomm error difference: [0.00212705]
ONNX ↔ Qualcomm error difference: [0.00212705]

STEP 18A — COMPLETE


In [76]:
# ==============================================================
# STEP 19 — DEPLOYMENT EVIDENCE SUMMARY
# ==============================================================

print("\n" + "=" * 70)
print("STEP 19 — SNAPDRAGON DEPLOYMENT EVIDENCE")
print("=" * 70)

print("\nTARGET DEVICE")
print("-" * 70)
print(target_device)

print("\nCOMPILE JOB")
print("-" * 70)
print("Job ID :", compile_job.job_id)
print("Status :", compile_job.get_status())

print("\nPROFILE JOB")
print("-" * 70)
print("Job ID :", profile_job.job_id)
print("Status :", profile_job.get_status())

print("\nINFERENCE JOB")
print("-" * 70)
print("Job ID :", inference_job.job_id)
print("Status :", inference_job.get_status())

print("\nPROFILE ARTIFACTS")
print("-" * 70)
print(profile_job.get_available_artifacts())

# --------------------------------------------------------------
# Load the existing profile JSON
# --------------------------------------------------------------

import json
import glob

profile_files = glob.glob(
    "guardian_profile/*.json"
)

print("\nProfile files:")
for f in profile_files:
    print(" ", f)

with open(profile_files[0], "r") as f:
    profile = json.load(f)

# --------------------------------------------------------------
# Execution summary
# --------------------------------------------------------------

execution_summary = profile["execution_summary"]

print("\nEXECUTION SUMMARY")
print("-" * 70)

for key, value in execution_summary.items():

    if key != "all_inference_times":
        print(f"{key}: {value}")

# --------------------------------------------------------------
# Execution detail
# --------------------------------------------------------------

print("\nEXECUTION DETAIL")
print("-" * 70)

for operation in profile["execution_detail"]:

    print(
        f"{operation['name']:20s} | "
        f"type={operation['type']:10s} | "
        f"compute_unit={operation['compute_unit']}"
    )

print("\n" + "=" * 70)
print("STEP 19 — COMPLETE")
print("=" * 70)


STEP 19 — SNAPDRAGON DEPLOYMENT EVIDENCE

TARGET DEVICE
----------------------------------------------------------------------
Device(name='Snapdragon X Elite CRD', os='', attributes=[])

COMPILE JOB
----------------------------------------------------------------------
Job ID : jglyxeel5
Status : JobStatus
---------
code    : SUCCESS
message : 


PROFILE JOB
----------------------------------------------------------------------
Job ID : jg9zm99vp
Status : JobStatus
---------
code    : SUCCESS
message : 


INFERENCE JOB
----------------------------------------------------------------------
Job ID : jpxl4ex1p
Status : JobStatus
---------
code    : SUCCESS
message : 


PROFILE ARTIFACTS
----------------------------------------------------------------------
[<JobArtifactType.DEVICE_LOG: 1>, <JobArtifactType.PROFILE_TRACE: 6>, <JobArtifactType.QHAS_SUMMARY: 8>]

Profile files:
  guardian_profile/future-model-jglyxeel5_jg9zm99vp_results.json

EXECUTION SUMMARY
-----------------------------

In [77]:
# ==============================================================
# STEP 20A — INSPECT CURRENT GUARDIAN RESULT
# ==============================================================

print("\n" + "=" * 70)
print("STEP 20A — CURRENT GUARDIAN RESULT INSPECTION")
print("=" * 70)

print("\nAvailable variables containing Guardian/detection results:\n")

for name in sorted(globals().keys()):

    name_lower = name.lower()

    if any(word in name_lower for word in [
        "guardian",
        "anomaly",
        "decision",
        "temporal",
        "volume",
        "context",
        "investigation",
        "result",
        "alert"
    ]):

        try:
            value = globals()[name]

            print(f"{name:35s} : {type(value).__name__}")

        except Exception:
            pass

print("\n" + "=" * 70)
print("STEP 20A — COMPLETE")
print("=" * 70)


STEP 20A — CURRENT GUARDIAN RESULT INSPECTION

Available variables containing Guardian/detection results:

GuardianAutoencoder                 : type
average_volume                      : float
full_guardian_analyze               : function
full_guardian_explain               : function
full_guardian_investigate           : function
full_guardian_investigate_timing    : function
full_guardian_investigate_volume    : function
guardian_tests                      : list
guardian_threshold                  : float
inference_results                   : InferenceJobResult
maximum_volume                      : int
minimum_volume                      : int
multiple_anomalies_result           : bool
new_context_event                   : dict
new_context_features                : list
new_context_result                  : bool
new_context_score                   : float
new_information_result              : bool
new_protocol_result                 : bool
new_relationship_result             : bo

In [80]:
# ==============================================================
# STEP 20B — INSPECT EXISTING GUARDIAN EXPLANATION
# ==============================================================

print("\n" + "=" * 70)
print("STEP 20B — EXISTING GUARDIAN EXPLANATION")
print("=" * 70)

print("\nCurrent result:")
print(result)

print("\n" + "-" * 70)

print("\nExisting full_guardian_explain() output:")

try:
    explanation_output = full_guardian_explain(
        results
    )

    print(explanation_output)

except Exception as e:

    print("Could not call full_guardian_explain(results)")
    print("Error:", type(e).__name__, "-", e)

print("\n" + "=" * 70)
print("STEP 20B — COMPLETE")
print("=" * 70)


STEP 20B — EXISTING GUARDIAN EXPLANATION

Current result:
PASS

----------------------------------------------------------------------

Existing full_guardian_explain() output:
Could not call full_guardian_explain(results)
Error: TypeError - full_guardian_explain() missing 1 required positional argument: 'result'

STEP 20B — COMPLETE


In [79]:
# ==============================================================
# STEP 20C — INSPECT GUARDIAN EXPLANATION FUNCTION
# ==============================================================

import inspect

print("\n" + "=" * 70)
print("STEP 20C — GUARDIAN EXPLANATION FUNCTION INSPECTION")
print("=" * 70)

print("\nFunction signature:")
print(inspect.signature(full_guardian_explain))

print("\nFunction source:")
print("-" * 70)

try:
    print(inspect.getsource(full_guardian_explain))
except Exception as e:
    print("Could not retrieve source.")
    print("Error:", type(e).__name__, "-", e)

print("\n" + "=" * 70)
print("STEP 20C — COMPLETE")
print("=" * 70)


STEP 20C — GUARDIAN EXPLANATION FUNCTION INSPECTION

Function signature:
(event, result)

Function source:
----------------------------------------------------------------------
def full_guardian_explain(event, result):

    print()
    print("-" * 70)

    print("SOURCE:      ", event["source"])
    print("DESTINATION: ", event["destination"])
    print("INFORMATION: ", event["information"])
    print(
        "PROTOCOL:    ",
        event["protocol"],
        "/",
        event["port"]
    )

    if "timestamp" in event:

        print(
            "TIMESTAMP:   ",
            event["timestamp"],
            "seconds"
        )

    if "bytes" in event:

        print(
            "MESSAGE SIZE:",
            event["bytes"],
            "bytes"
        )


    print()
    print(
        "AI SCORE:    ",
        round(result["score"], 6)
    )

    print(
        "THRESHOLD:   ",
        round(guardian_threshold, 6)
    )

    print(
        "AI DECISION: ",
        "SUSPICIOUS"
   

In [81]:
# ==============================================================
# STEP 20D — INSPECT GUARDIAN EVENT + RESULT
# ==============================================================

print("\n" + "=" * 70)
print("STEP 20D — GUARDIAN EVENT + RESULT")
print("=" * 70)

print("\nRESULT OBJECT")
print("-" * 70)

print("Type:", type(result).__name__)
print(result)

print("\n" + "-" * 70)

print("\nRESULTS OBJECT")
print("-" * 70)

print("Type:", type(results).__name__)
print(results)

print("\n" + "-" * 70)

print("\nNEW CONTEXT EVENT")
print("-" * 70)

print("Type:", type(new_context_event).__name__)
print(new_context_event)

print("\n" + "=" * 70)
print("STEP 20D — COMPLETE")
print("=" * 70)


STEP 20D — GUARDIAN EVENT + RESULT

RESULT OBJECT
----------------------------------------------------------------------
Type: str
PASS

----------------------------------------------------------------------

RESULTS OBJECT
----------------------------------------------------------------------
Type: dict
{'Test 1 - Known normal': True, 'Test 2 - Reverse direction': True, 'Test 3 - New information': True, 'Test 4 - New protocol': True, 'Test 5 - New relationship': True, 'Test 6 - New context': True, 'Test 7 - Unknown source': True, 'Test 8 - Unknown destination': True, 'Test 9 - Multiple anomalies': True}

----------------------------------------------------------------------

NEW CONTEXT EVENT
----------------------------------------------------------------------
Type: dict
{'source': 'PLC-02', 'destination': 'HIST-01', 'information': 'process_read', 'protocol': 'OPC-UA', 'port': 4840}

STEP 20D — COMPLETE


In [84]:
print("=" * 70)
print("NEW CONTEXT RESULT")
print("=" * 70)

print(type(new_context_result).__name__)
print(new_context_result)
import inspect

print(inspect.getsource(full_guardian_analyze))

print("=" * 70)
print("ACTUAL GUARDIAN ANALYSIS")
print("=" * 70)

actual_analysis = full_guardian_analyze(new_context_event)

print(type(actual_analysis).__name__)
print(actual_analysis)

NEW CONTEXT RESULT
bool
True
def full_guardian_analyze(event):

    # --------------------------------------------------------------
    # AI
    # --------------------------------------------------------------

    score = (
        pytorch_guardian_reconstruction_error(
            event
        )
    )

    ai_suspicious = (
        score > guardian_threshold
    )


    # --------------------------------------------------------------
    # Context
    # --------------------------------------------------------------

    context_evidence, context = (
        full_guardian_investigate(event)
    )


    # --------------------------------------------------------------
    # Temporal
    # --------------------------------------------------------------

    timing_evidence = (
        full_guardian_investigate_timing(
            event
        )
    )


    # --------------------------------------------------------------
    # Volume
    # -----------------------------------------------

In [86]:
def full_guardian_explain(event, result):

    print()
    print("=" * 70)
    print("OT GUARDIAN SECURITY ALERT")
    print("=" * 70)

    # --------------------------------------------------------------
    # STATUS
    # --------------------------------------------------------------

    if result["status"] == "SUSPICIOUS":
        print()
        print("STATUS")
        print("-" * 70)
        print("⚠ SUSPICIOUS COMMUNICATION DETECTED")
    else:
        print()
        print("STATUS")
        print("-" * 70)
        print("✓ COMMUNICATION BEHAVIOUR NORMAL")

    # --------------------------------------------------------------
    # EVENT
    # --------------------------------------------------------------

    print()
    print("EVENT")
    print("-" * 70)

    print("Source:        ", event["source"])
    print("Destination:   ", event["destination"])
    print("Operation:     ", event["information"])
    print("Protocol:      ", event["protocol"])
    print("Port:          ", event["port"])

    if "timestamp" in event:
        print("Timestamp:     ", event["timestamp"], "seconds")

    if "bytes" in event:
        print("Message size:  ", event["bytes"], "bytes")

    # --------------------------------------------------------------
    # WHAT GUARDIAN OBSERVED
    # --------------------------------------------------------------

    print()
    print("WHAT GUARDIAN OBSERVED")
    print("-" * 70)

    if result["status"] == "SUSPICIOUS":

        print(
            "A communication was observed between",
            event["source"],
            "and",
            event["destination"] + "."
        )

        if result["context_evidence"]:
            print(
                "Guardian identified an unusual communication context."
            )

        if result["timing_evidence"]:
            print(
                "Guardian identified unusual temporal behaviour."
            )

        if result["volume_evidence"]:
            print(
                "Guardian identified unusual communication volume."
            )

    else:

        print(
            "The observed communication matches the behaviour",
            "expected by the Guardian baseline."
        )

    # --------------------------------------------------------------
    # WHY WAS IT FLAGGED?
    # --------------------------------------------------------------

    print()
    print("WHY WAS IT FLAGGED?")
    print("-" * 70)

    if result["evidence"]:

        for item in result["evidence"]:
            print("•", item)

    else:

        print("• No contextual, temporal, or volume anomaly detected.")

    # --------------------------------------------------------------
    # ADDITIONAL CHECKS
    # --------------------------------------------------------------

    print()
    print("ADDITIONAL CHECKS")
    print("-" * 70)

    print(
        "AI anomaly:          ",
        "YES" if result["ai_suspicious"] else "NO"
    )

    print(
        "Context anomaly:     ",
        "YES" if result["context_evidence"] else "NO"
    )

    print(
        "Temporal anomaly:    ",
        "YES" if result["timing_evidence"] else "NO"
    )

    print(
        "Volume anomaly:      ",
        "YES" if result["volume_evidence"] else "NO"
    )

    # --------------------------------------------------------------
    # AI ASSESSMENT
    # --------------------------------------------------------------

    print()
    print("AI ASSESSMENT")
    print("-" * 70)

    print(
        "Reconstruction error:",
        round(result["score"], 6)
    )

    print(
        "Guardian threshold:  ",
        round(guardian_threshold, 6)
    )

    if result["ai_suspicious"]:

        print()
        print(
            "The AI model considers this communication behaviour anomalous."
        )

    else:

        print()
        print(
            "The AI model considers this communication behaviour normal."
        )

    # --------------------------------------------------------------
    # CONTEXT
    # --------------------------------------------------------------

    print()
    print("COMMUNICATION CONTEXT")
    print("-" * 70)

    context = result["context"]

    print(
        "Relationship known:          ",
        "YES" if context["relationship_known"] else "NO"
    )

    print(
        "Direction known:             ",
        "YES" if context["direction_known"] else "NO"
    )

    print(
        "Information known globally:  ",
        "YES" if context["information_known_globally"] else "NO"
    )

    print(
        "Information valid for context:",
        "YES" if context["information_known_for_direction"] else "NO"
    )

    print(
        "Protocol known globally:     ",
        "YES" if context["protocol_known_globally"] else "NO"
    )

    print(
        "Protocol valid for context:  ",
        "YES" if context["protocol_known_for_direction"] else "NO"
    )

    # --------------------------------------------------------------
    # HUMAN ACTION
    # --------------------------------------------------------------

    print()
    print("RECOMMENDED HUMAN ACTION")
    print("-" * 70)

    if result["status"] == "SUSPICIOUS":

        print(
            "Verify whether this communication is expected for",
            "the current industrial process."
        )

        print(
            "If unexpected, investigate the communication and",
            "determine whether the change was authorized."
        )

    else:

        print(
            "No immediate investigation is indicated by the",
            "current Guardian assessment."
        )

    # --------------------------------------------------------------
    # AUTOMATIC INTERVENTION
    # --------------------------------------------------------------

    print()
    print("AUTOMATIC INTERVENTION")
    print("-" * 70)

    print(
        "None. Guardian is operating in monitoring and",
        "notification mode."
    )

    print()
    print("=" * 70)

In [87]:
actual_analysis = full_guardian_analyze(new_context_event)

full_guardian_explain(
    new_context_event,
    actual_analysis
)


OT GUARDIAN SECURITY ALERT

STATUS
----------------------------------------------------------------------
⚠ SUSPICIOUS COMMUNICATION DETECTED

EVENT
----------------------------------------------------------------------
Source:         PLC-02
Destination:    HIST-01
Operation:      process_read
Protocol:       OPC-UA
Port:           4840

WHAT GUARDIAN OBSERVED
----------------------------------------------------------------------
A communication was observed between PLC-02 and HIST-01.
Guardian identified an unusual communication context.

WHY WAS IT FLAGGED?
----------------------------------------------------------------------
• Source device is unknown
• Destination device is unknown
• Known information type used in a new communication context

ADDITIONAL CHECKS
----------------------------------------------------------------------
AI anomaly:           YES
Context anomaly:      YES
Temporal anomaly:     NO
Volume anomaly:       NO

AI ASSESSMENT
----------------------------------